
# Decoder-Only Transformer and KV Cache in PyTorch for Interview Prep

This notebook is a **teaching-first, interview-focused walkthrough** of a **decoder-only Transformer** with a **working KV cache**.

It is designed for:
- LLM inference interviews
- LLM training interviews
- LLM infra / systems interviews
- candidates who want both **intuition** and **working PyTorch code**

## What this notebook covers

By the end, you will have:
1. a decoder-only GPT-style model in PyTorch
2. causal self-attention
3. learned token + position embeddings
4. residual blocks with MLPs and layer norm
5. a tiny training loop
6. generation **without** KV cache
7. generation **with** KV cache
8. a correctness check showing cached and uncached generation agree
9. a small timing comparison
10. interview-style exercises with **hints and answers**

## Why this notebook matters

For modern LLM roles, you usually need to understand:
- why decoder-only models are the common LLM backbone
- how autoregressive generation differs from training
- why causal masking matters
- why inference is slower than training across time steps
- what a KV cache stores
- why KV cache speeds up autoregressive inference
- why KV cache is also a systems problem: it trades compute for memory



## Scope and philosophy

This notebook optimizes for:
- **clear shapes**
- **clean mental models**
- **heavily commented code**
- **interview usefulness**

It does **not** optimize for maximum throughput or production kernel efficiency.

For production systems, you would usually rely on:
- optimized attention kernels
- mixed precision
- fused ops
- serving frameworks such as vLLM
- more advanced positional methods such as RoPE

Here, the goal is to deeply understand the **ideas**.



## High-level mental model

A decoder-only Transformer does two big things over and over:

1. **Mix information across tokens** with **causal self-attention**
2. **Transform features** with a feed-forward network

Each layer repeats:
- layer norm
- causal self-attention
- residual connection
- layer norm
- feed-forward network
- residual connection

The model is "decoder-only" because it has:
- no encoder stack
- no cross-attention
- only masked self-attention

That matches the basic structure of GPT-style LLMs.


In [ ]:

# ============================================================
# Imports and device setup
# ============================================================
# We keep this notebook CPU/GPU aware so it runs on:
# - Google Colab with a GPU
# - a local CPU-only machine
#
# Most of the model code stays the same in both cases.
# The main change is where tensors and modules live.

import math
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility matters for debugging and interview prep.
SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

# This can improve some matrix multiply choices on supported setups.
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. The notebook still works on CPU.")



## 1. Tiny vocabulary and toy task

We use a very small synthetic task so the notebook stays focused on the architecture.

### Task: counting continuation
Each training sequence looks like:

`<bos> 3 4 5 6 7 <eos>`

The model is trained with next-token prediction:
- input side: everything except the last token
- target side: everything except the first token

This is a nice decoder-only task because:
- it is easy to batch
- next-token prediction is natural
- generation looks like real autoregressive decoding

We are **not** trying to solve a hard benchmark.
We are learning the mechanics.


In [ ]:

# ============================================================
# Tiny vocabulary
# ============================================================
# Token IDs:
# 0 -> padding
# 1 -> beginning of sequence
# 2 -> end of sequence
# 3..12 -> digits 0..9

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
DIGIT_OFFSET = 3
VOCAB_SIZE = 13

ID_TO_TOKEN: Dict[int, str] = {
    PAD_ID: "<pad>",
    BOS_ID: "<bos>",
    EOS_ID: "<eos>",
}

for digit in range(10):
    ID_TO_TOKEN[DIGIT_OFFSET + digit] = str(digit)

TOKEN_TO_ID: Dict[str, int] = {token: idx for idx, token in ID_TO_TOKEN.items()}

def token_ids_to_text(token_ids: List[int]) -> str:
    """Render token IDs in a human-friendly way for debugging."""
    pieces: List[str] = []
    for token_id in token_ids:
        token = ID_TO_TOKEN.get(int(token_id), f"<?{int(token_id)}>")
        if token != "<pad>":
            pieces.append(token)
    return " ".join(pieces)

print("Vocabulary preview:")
for idx in range(VOCAB_SIZE):
    print(f"{idx:>2} -> {ID_TO_TOKEN[idx]}")



## 2. Decoder-only attention: the causal rule

A decoder-only model predicts token `t` using only:
- tokens before `t`
- token `t` itself (depending on training formulation)

It must **not** look into the future.

That is why decoder-only attention needs a **causal mask**.

### Interview intuition
If token 5 could attend to token 8 during training, the model would be cheating.
It would learn from information that is unavailable at inference time.


In [ ]:

# ============================================================
# Causal mask helper
# ============================================================
# We create a boolean mask where:
# True  = this key position is allowed
# False = this key position must be blocked
#
# Why do we need a helper that knows about "past_len"?
# Because when we use a KV cache, the current query token may be
# attending over a key sequence that is longer than the current
# input chunk. The cache already contains earlier keys/values.

def make_causal_mask(
    query_len: int,
    key_len: int,
    device: torch.device,
    past_len: int = 0,
) -> torch.Tensor:
    """
    Build a causal mask with shape [1, 1, query_len, key_len].

    query positions correspond to absolute positions:
        past_len, past_len + 1, ..., past_len + query_len - 1

    key positions correspond to absolute positions:
        0, 1, 2, ..., key_len - 1
    """
    query_positions = torch.arange(
        past_len, past_len + query_len, device=device
    )[:, None]
    key_positions = torch.arange(key_len, device=device)[None, :]

    keep = key_positions <= query_positions
    return keep.unsqueeze(0).unsqueeze(0)

# Visualize one full causal mask.
demo_mask = make_causal_mask(query_len=6, key_len=6, device=torch.device("cpu"))

plt.figure(figsize=(4, 3))
plt.imshow(demo_mask[0, 0].numpy(), aspect="auto")
plt.title("Causal mask")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.colorbar()
plt.show()



## 3. Embeddings and positional information

The model sees integer token IDs, but neural networks operate on vectors.

So we first use:
- **token embeddings** to map token IDs into dense vectors
- **position embeddings** to tell the model where each token sits in the sequence

### Why learned positions here?
Many GPT-style models use learned or rotary positional information.
For this notebook, learned position embeddings keep the implementation short and interview-friendly.



## 4. Causal self-attention and the KV cache mental model

### Normal self-attention without cache
At every generation step, if you feed the whole prefix back into the model, the model recomputes:
- all key vectors for old tokens
- all value vectors for old tokens
- all intermediate states for old tokens

That wastes work.

### KV cache idea
Store the **K** and **V** tensors from earlier tokens for each layer.

Then, at the next decoding step:
- compute Q/K/V only for the **new** token
- append the new K/V to the cache
- attend over:
  - cached past K/V
  - new K/V

This avoids recomputing old K/V.

### Important nuance
KV caching avoids recomputing old projections and hidden-state paths, but the new token still attends over the full history.
So generation does **not** become O(1) overall with respect to context length.
The attention work for the new token still grows with context.


In [ ]:

# ============================================================
# Causal self-attention with optional KV cache
# ============================================================
# This module is the heart of the notebook.
#
# It supports two modes:
#
# 1) Normal full-sequence mode (training, or generation without cache)
# 2) Incremental cached mode (generation with cache)
#
# We use torch.nn.functional.scaled_dot_product_attention because it
# mirrors how modern PyTorch expresses attention cleanly.

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Standard Q, K, V projections.
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        # Output projection back into model dimension.
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Convert [batch, seq_len, d_model]
        into     [batch, heads, seq_len, head_dim]
        """
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 2)
        return x

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Convert [batch, heads, seq_len, head_dim]
        back into [batch, seq_len, d_model]
        """
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        x = x.view(batch_size, seq_len, self.d_model)
        return x

    def forward(
        self,
        x: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        """
        Args:
            x:
                [batch, seq_len, d_model]
            past_kv:
                Optional tuple of cached tensors:
                (
                    past_keys,   # [batch, heads, past_len, head_dim]
                    past_values  # [batch, heads, past_len, head_dim]
                )
            use_cache:
                Whether to return the updated cache.

        Returns:
            attention_output:
                [batch, seq_len, d_model]
            new_cache:
                Optional updated (K, V) tuple
        """
        # Project the current input chunk into Q, K, V.
        q = self._split_heads(self.q_proj(x))
        k_new = self._split_heads(self.k_proj(x))
        v_new = self._split_heads(self.v_proj(x))

        past_len = 0

        # If a cache exists, append the new K/V to the old K/V.
        if past_kv is not None:
            past_k, past_v = past_kv
            past_len = past_k.size(-2)

            k = torch.cat([past_k, k_new], dim=-2)
            v = torch.cat([past_v, v_new], dim=-2)
        else:
            k = k_new
            v = v_new

        new_cache = (k, v) if use_cache else None

        # There are two important cases:
        #
        # Case A: no cache yet
        #   This is training or prompt prefill.
        #   The current chunk is a full sequence, so standard causal
        #   masking applies over that chunk.
        #
        # Case B: cache already exists
        #   This is incremental decode.
        #   The query is only the new token(s), but keys/values include
        #   all the old cached tokens plus the new ones.
        #   We build an explicit boolean mask.
        if past_len == 0:
            attention_out = F.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=None,
                dropout_p=self.dropout_p if self.training else 0.0,
                is_causal=True,
            )
        else:
            attn_mask = make_causal_mask(
                query_len=x.size(1),
                key_len=k.size(-2),
                device=x.device,
                past_len=past_len,
            )

            attention_out = F.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=attn_mask,
                dropout_p=self.dropout_p if self.training else 0.0,
                is_causal=False,
            )

        # Merge heads and project back into model dimension.
        attention_out = self._merge_heads(attention_out)
        attention_out = self.resid_dropout(self.out_proj(attention_out))

        return attention_out, new_cache



## 5. Feed-forward layer and decoder block

A decoder block alternates:
- causal self-attention
- feed-forward network

with:
- residual connections
- layer normalization

This is the same high-level pattern you saw in the full Transformer, but here we only keep the decoder-style path.


In [ ]:

# ============================================================
# Feed-forward network and decoder block
# ============================================================

class FeedForward(nn.Module):
    def __init__(self, d_model: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()

        self.fc1 = nn.Linear(d_model, ff_dim)
        self.fc2 = nn.Linear(ff_dim, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # GELU is common in GPT-style models, so we use it here.
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model=d_model, ff_dim=ff_dim, dropout=dropout)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        # Pre-norm attention
        attn_out, new_cache = self.attn(self.ln1(x), past_kv=past_kv, use_cache=use_cache)
        x = x + self.dropout(attn_out)

        # Pre-norm feed-forward
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)

        return x, new_cache



## 6. Full decoder-only Transformer

This model includes:
- token embeddings
- learned positional embeddings
- stacked decoder blocks
- final layer norm
- language-model head

### Weight tying
We tie the output projection weights to the token embedding weights.
That is common in language models and reduces parameter count.


In [ ]:

# ============================================================
# Full decoder-only Transformer
# ============================================================

class DecoderOnlyTransformer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 32,
        num_heads: int = 4,
        ff_dim: int = 64,
        num_layers: int = 2,
        dropout: float = 0.1,
        max_len: int = 128,
    ):
        super().__init__()

        self.d_model = d_model
        self.max_len = max_len

        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.embed_dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            DecoderBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.final_ln = nn.LayerNorm(d_model)

        # Tie the LM head weights to the token embedding table.
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embed.weight

    def embed(self, input_ids: torch.Tensor, start_pos: int = 0) -> torch.Tensor:
        """
        Embed tokens and add learned position embeddings.

        start_pos matters for cached generation:
        - prompt prefill starts at position 0
        - later decode steps continue from the cached length
        """
        batch_size, seq_len = input_ids.shape

        positions = torch.arange(start_pos, start_pos + seq_len, device=input_ids.device)

        x = self.token_embed(input_ids) * math.sqrt(self.d_model)
        x = x + self.pos_embed(positions)[None, :, :]
        x = self.embed_dropout(x)
        return x

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Standard full forward pass with no cache.

        input_ids shape:
            [batch, seq_len]

        returns logits shape:
            [batch, seq_len, vocab_size]
        """
        x = self.embed(input_ids, start_pos=0)

        for layer in self.layers:
            x, _ = layer(x, past_kv=None, use_cache=False)

        x = self.final_ln(x)
        logits = self.lm_head(x)
        return logits

    def forward_with_cache(
        self,
        input_ids: torch.Tensor,
        kv_cache: Optional[List[Optional[Tuple[torch.Tensor, torch.Tensor]]]] = None,
        use_cache: bool = True,
    ) -> Tuple[torch.Tensor, List[Optional[Tuple[torch.Tensor, torch.Tensor]]]]:
        """
        Forward pass that supports a per-layer KV cache.

        kv_cache is a list with one entry per layer.
        Each entry is either:
            None
        or:
            (keys, values)

        This method is used for:
        - prompt prefill
        - incremental decoding
        """
        if kv_cache is None:
            kv_cache = [None] * len(self.layers)
            start_pos = 0
        else:
            first_layer_cache = kv_cache[0]
            start_pos = 0 if first_layer_cache is None else first_layer_cache[0].size(-2)

        x = self.embed(input_ids, start_pos=start_pos)

        new_cache: List[Optional[Tuple[torch.Tensor, torch.Tensor]]] = []
        for layer, past_layer_kv in zip(self.layers, kv_cache):
            x, layer_cache = layer(x, past_kv=past_layer_kv, use_cache=use_cache)
            new_cache.append(layer_cache)

        x = self.final_ln(x)
        logits = self.lm_head(x)
        return logits, new_cache

    @torch.inference_mode()
    def generate_without_cache(self, prompt_ids: torch.Tensor, max_new_tokens: int = 8) -> torch.Tensor:
        """
        Autoregressive generation without KV cache.

        This re-feeds the whole growing prefix into the model at every step.
        It is simple but wasteful.
        """
        self.eval()

        generated = prompt_ids.clone()

        for _ in range(max_new_tokens):
            logits = self.forward(generated)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)

            if bool((next_token == EOS_ID).all()):
                break

        return generated

    @torch.inference_mode()
    def generate_with_cache(self, prompt_ids: torch.Tensor, max_new_tokens: int = 8) -> torch.Tensor:
        """
        Autoregressive generation with KV cache.

        Step 1: prefill the prompt once
        Step 2: decode one token at a time using cached K/V from each layer
        """
        self.eval()

        # Prompt prefill: run the whole prompt once and build caches.
        logits, kv_cache = self.forward_with_cache(prompt_ids, kv_cache=None, use_cache=True)

        generated = prompt_ids.clone()

        # First generated token comes from the last prompt position.
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)

        if bool((next_token == EOS_ID).all()):
            return generated

        # Incremental decode: only feed the newest token each step.
        for _ in range(max_new_tokens - 1):
            logits, kv_cache = self.forward_with_cache(
                next_token,
                kv_cache=kv_cache,
                use_cache=True,
            )
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)

            if bool((next_token == EOS_ID).all()):
                break

        return generated



## 7. Toy batch generator

We generate short counting sequences like:

`<bos> 4 5 6 7 8 <eos>`

Then train next-token prediction:
- input: `<bos> 4 5 6 7 8`
- target: `4 5 6 7 8 <eos>`

This is the most natural training setup for a decoder-only language model.


In [ ]:

# ============================================================
# Toy batch generator for next-token prediction
# ============================================================

def make_counting_batch(
    batch_size: int,
    min_digits: int = 4,
    max_digits: int = 7,
    device: torch.device = torch.device("cpu"),
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Build a batch of simple counting sequences.

    Example full sequence:
        <bos> 3 4 5 6 <eos>

    For language-model training:
        input_ids  = <bos> 3 4 5 6
        target_ids = 3 4 5 6 <eos>
    """
    rows: List[List[int]] = []
    max_len = 0

    for _ in range(batch_size):
        start_digit = random.randint(0, 9)
        seq_len = random.randint(min_digits, max_digits)

        digits = [DIGIT_OFFSET + ((start_digit + i) % 10) for i in range(seq_len)]

        full_sequence = [BOS_ID] + digits + [EOS_ID]
        rows.append(full_sequence)

        max_len = max(max_len, len(full_sequence))

    padded = [
        row + [PAD_ID] * (max_len - len(row))
        for row in rows
    ]

    full = torch.tensor(padded, dtype=torch.long, device=device)

    input_ids = full[:, :-1]
    target_ids = full[:, 1:]

    return input_ids, target_ids

# Preview a sample batch
preview_inputs, preview_targets = make_counting_batch(batch_size=3, device=device)

print("Input batch:")
print(preview_inputs)
print("\nTarget batch:")
print(preview_targets)

print("\nReadable sample 0:")
print("input  ->", token_ids_to_text(preview_inputs[0].tolist()))
print("target ->", token_ids_to_text(preview_targets[0].tolist()))



## 8. Build the model and run a forward pass

We intentionally use a small model so the notebook stays runnable in Colab and CPU environments.


In [ ]:

# ============================================================
# Instantiate a small decoder-only model
# ============================================================

model = DecoderOnlyTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout=0.1,
    max_len=64,
).to(device)

num_parameters = sum(p.numel() for p in model.parameters())

print(model)
print(f"\nTrainable parameter count: {num_parameters:,}")

# Forward-pass smoke test
input_ids, target_ids = make_counting_batch(batch_size=4, device=device)
logits = model(input_ids)

print("\nInput shape: ", tuple(input_ids.shape))
print("Target shape:", tuple(target_ids.shape))
print("Logits shape:", tuple(logits.shape))

assert logits.shape[:2] == input_ids.shape
assert logits.size(-1) == VOCAB_SIZE



## 9. Training loop

The training objective is plain next-token prediction with cross-entropy loss.

### Why `ignore_index=PAD_ID`?
The padded positions are not real targets.  
We added them only so variable-length sequences can share a batch tensor shape.

### Why are the default training runs short?
This notebook is meant to run quickly.  
If you are on Colab GPU, you can safely increase the number of steps.


In [ ]:

# ============================================================
# Tiny training loop
# ============================================================

@dataclass
class TrainConfig:
    batch_size: int = 6
    learning_rate: float = 2e-3
    grad_clip_norm: float = 1.0
    cpu_steps: int = 3
    gpu_steps: int = 6


train_config = TrainConfig()

# Keep the default run short so the notebook stays responsive.
num_train_steps = train_config.gpu_steps if device.type == "cuda" else train_config.cpu_steps

optimizer = torch.optim.Adam(model.parameters(), lr=train_config.learning_rate)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

def train_one_step(model: nn.Module) -> float:
    """Run one next-token prediction step."""
    model.train()

    input_ids, target_ids = make_counting_batch(
        batch_size=train_config.batch_size,
        device=device,
    )

    logits = model(input_ids)

    # CrossEntropyLoss expects:
    #   logits  -> [N, C]
    #   targets -> [N]
    #
    # So we flatten batch and time together.
    loss = loss_fn(
        logits.reshape(-1, VOCAB_SIZE),
        target_ids.reshape(-1),
    )

    optimizer.zero_grad()
    loss.backward()

    # Gradient clipping is a common stability trick.
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=train_config.grad_clip_norm)

    optimizer.step()
    return float(loss.item())

loss_history: List[float] = []

train_start = time.perf_counter()
for step_idx in range(num_train_steps):
    loss_value = train_one_step(model)
    loss_history.append(loss_value)
train_elapsed = time.perf_counter() - train_start

print(f"Ran {num_train_steps} training steps on {device}.")
print(f"Elapsed time: {train_elapsed:.3f} seconds")
print("Loss history:", [round(v, 4) for v in loss_history])

plt.figure(figsize=(6, 3))
plt.plot(loss_history, marker="o")
plt.title("Training loss (short demo run)")
plt.xlabel("Step")
plt.ylabel("Cross-entropy loss")
plt.grid(True, alpha=0.3)
plt.show()



## 10. Generation without cache

This is the simplest autoregressive loop:

1. feed the prompt into the model
2. take the last-position logits
3. choose the next token
4. append it to the prompt
5. run the whole enlarged sequence again
6. repeat

### Why is this wasteful?
Because the model keeps recomputing hidden states, K, and V for earlier tokens that never changed.


In [ ]:

# ============================================================
# Generation demo: without cache
# ============================================================

model.eval()

prompt = torch.tensor(
    [[BOS_ID, DIGIT_OFFSET + 2, DIGIT_OFFSET + 3, DIGIT_OFFSET + 4]],
    dtype=torch.long,
    device=device,
)

generated_no_cache = model.generate_without_cache(prompt, max_new_tokens=5)

print("Prompt only:          ", token_ids_to_text(prompt[0].tolist()))
print("Generated no-cache:   ", token_ids_to_text(generated_no_cache[0].tolist()))



## 11. Why KV cache helps

Suppose prompt length is `P` and you generate `N` new tokens.

### Without cache
You repeatedly process:
- prompt of length `P`
- then `P + 1`
- then `P + 2`
- ...
- then `P + N - 1`

So the model keeps revisiting old tokens.

### With cache
You:
- process the prompt once (**prefill**)
- then for each new token, process only the **new token** through the decoder stack
- reuse old per-layer K/V from the cache

### Important serving vocabulary
- **Prefill** = run the prompt and build initial cache
- **Decode** = generate new tokens one step at a time using the cache


In [ ]:

# ============================================================
# Simple intuition calculator: token positions reprocessed
# ============================================================
# This is NOT an exact FLOP count.
# It is just a clean way to illustrate repeated work.

def count_positions_processed_without_cache(prompt_len: int, new_tokens: int) -> int:
    total = 0
    current_len = prompt_len
    for _ in range(new_tokens):
        total += current_len
        current_len += 1
    return total

def count_positions_processed_with_cache(prompt_len: int, new_tokens: int) -> int:
    # One full prompt prefill, then 1 new token each decode step.
    return prompt_len + new_tokens

prompt_len = 6
new_tokens = 8

no_cache_positions = count_positions_processed_without_cache(prompt_len, new_tokens)
with_cache_positions = count_positions_processed_with_cache(prompt_len, new_tokens)

print(f"Prompt length: {prompt_len}")
print(f"New tokens:    {new_tokens}")
print(f"Without cache, positions reprocessed: {no_cache_positions}")
print(f"With cache, positions reprocessed:    {with_cache_positions}")



## 12. Generation with cache

Now we use the `forward_with_cache(...)` path.

### Cache layout
We store one cache entry **per layer**.

Each entry is:
- keys  shape: `[batch, heads, cached_len, head_dim]`
- values shape: `[batch, heads, cached_len, head_dim]`

### Why per-layer?
Each decoder block has its **own** attention projections and therefore its own K/V tensors.


In [ ]:

# ============================================================
# Generation demo: with cache
# ============================================================

generated_with_cache = model.generate_with_cache(prompt, max_new_tokens=5)

print("Prompt only:          ", token_ids_to_text(prompt[0].tolist()))
print("Generated with-cache: ", token_ids_to_text(generated_with_cache[0].tolist()))



## 13. Cached vs uncached correctness check

Before trusting a cache implementation, you should check that it matches the uncached behavior.

We do two checks:
1. **Prefill equivalence**  
   Full forward pass on a prompt should match `forward_with_cache(prompt, None)`
2. **Incremental equivalence**  
   One-step cached decoding should match the last-token logits from a full recomputed forward pass on the extended sequence


In [ ]:

# ============================================================
# Correctness checks for the cache path
# ============================================================

model.eval()

test_prompt = torch.tensor(
    [[BOS_ID, DIGIT_OFFSET + 1, DIGIT_OFFSET + 2]],
    dtype=torch.long,
    device=device,
)

# Check 1: prefill equivalence
full_logits = model(test_prompt)
prefill_logits, kv_cache = model.forward_with_cache(test_prompt, kv_cache=None, use_cache=True)

prefill_equal = torch.allclose(full_logits, prefill_logits, atol=1e-5, rtol=1e-4)
print("Prefill logits match full forward:", prefill_equal)

# Check 2: incremental decode equivalence
next_token = full_logits[:, -1, :].argmax(dim=-1, keepdim=True)

full_extended = torch.cat([test_prompt, next_token], dim=1)
full_extended_logits = model(full_extended)[:, -1, :]

incremental_logits, kv_cache = model.forward_with_cache(
    next_token,
    kv_cache=kv_cache,
    use_cache=True,
)

incremental_equal = torch.allclose(
    full_extended_logits,
    incremental_logits[:, -1, :],
    atol=1e-5,
    rtol=1e-4,
)

print("Incremental logits match full recomputation:", incremental_equal)

# Also compare whole generated sequences.
no_cache_seq = model.generate_without_cache(test_prompt, max_new_tokens=4)
with_cache_seq = model.generate_with_cache(test_prompt, max_new_tokens=4)
sequence_equal = torch.equal(no_cache_seq, with_cache_seq)

print("Generated sequences match:", sequence_equal)
print("No-cache sequence: ", token_ids_to_text(no_cache_seq[0].tolist()))
print("With-cache sequence:", token_ids_to_text(with_cache_seq[0].tolist()))



## 14. Inspect cache shapes

Cache shape inspection is a very common interview question.

If the model has:
- `B` batch elements
- `H` attention heads
- cached sequence length `T`
- `D_h` head dimension

then each layer stores:
- keys:   `[B, H, T, D_h]`
- values: `[B, H, T, D_h]`


In [ ]:

# ============================================================
# Inspect cache tensor shapes
# ============================================================

for layer_idx, layer_cache in enumerate(kv_cache):
    if layer_cache is None:
        print(f"Layer {layer_idx}: cache is None")
        continue

    keys, values = layer_cache
    print(f"Layer {layer_idx}")
    print("  K shape:", tuple(keys.shape))
    print("  V shape:", tuple(values.shape))



## 15. Tiny timing comparison

This timing demo is intentionally small.

### Important caveat
Because this notebook uses:
- a tiny model
- short sequences
- Python-level loops

the measured speedup may be:
- small
- noisy
- sometimes unimpressive on CPU

That does **not** mean KV cache is unimportant.
It just means the notebook is a teaching artifact, not a production benchmark.

In real LLM serving, KV cache is one of the most important inference optimizations.


In [ ]:

# ============================================================
# Tiny timing benchmark: generate with vs without cache
# ============================================================

def benchmark_generation(model: DecoderOnlyTransformer, prompt: torch.Tensor, repeats: int = 1) -> Dict[str, float]:
    # Warm-up runs
    _ = model.generate_without_cache(prompt, max_new_tokens=3)
    _ = model.generate_with_cache(prompt, max_new_tokens=3)

    if prompt.device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(repeats):
        _ = model.generate_without_cache(prompt, max_new_tokens=3)
    if prompt.device.type == "cuda":
        torch.cuda.synchronize()
    end = time.perf_counter()
    no_cache_avg = (end - start) / repeats

    start = time.perf_counter()
    for _ in range(repeats):
        _ = model.generate_with_cache(prompt, max_new_tokens=3)
    if prompt.device.type == "cuda":
        torch.cuda.synchronize()
    end = time.perf_counter()
    with_cache_avg = (end - start) / repeats

    return {
        "no_cache_avg_sec": no_cache_avg,
        "with_cache_avg_sec": with_cache_avg,
    }

timing_prompt = torch.tensor(
    [[BOS_ID, DIGIT_OFFSET + 5, DIGIT_OFFSET + 6, DIGIT_OFFSET + 7, DIGIT_OFFSET + 8]],
    dtype=torch.long,
    device=device,
)

timing_results = benchmark_generation(model, timing_prompt, repeats=1)

print("Average generation times:")
for name, value in timing_results.items():
    print(f"  {name}: {value:.6f} sec")



## 16. Decoder-only training vs inference

This is a high-frequency interview topic.

### Training
- We usually feed a full sequence.
- The model predicts every next token in parallel across positions.
- We use a causal mask so each position only sees the left context.

### Inference
- Future tokens are unknown.
- We generate one token at a time.
- That makes decoding much less parallel across time.
- KV cache helps by avoiding repeated work for old tokens.

This is one reason serving LLMs is hard:
**training and inference have very different runtime behavior**.



## 17. Systems and infra angle: why KV cache matters beyond model code

KV cache is not just a modeling trick. It is a systems concern.

### Benefits
- lower repeated compute during decoding
- much better autoregressive inference efficiency

### Costs
- the cache consumes memory
- longer contexts need larger caches
- more concurrent requests increase memory pressure

### Serving consequence
In a real inference server, you have to think about:
- how much GPU memory KV cache occupies
- how batching interacts with cache memory
- how long prompts affect prefill time
- how long generations affect decode time

That is why KV cache is central to LLM inference engineering.



## 18. Interview-style exercises

### Exercise 1 — Why does a decoder-only model need a causal mask?

**Hint:**  
Think about what would happen during training if token position `t` could attend to token position `t+1`.



### Exercise 1 — Answer

A decoder-only model is trained for next-token prediction.

If token position `t` could see token `t+1`, the model would be cheating:
it would use future information that is unavailable during real generation.

The causal mask ensures training matches the left-to-right inference setting.


## 21. Real Model Memory Math: Why KV Cache Is A Systems Problem

This is one of the places where exact arithmetic matters.

**Rule for this section:**
- when I give a tensor-shape memory calculation, it is exact for the stated assumptions
- when I later discuss latency or system behavior, I will label it as **illustrative** or **toy napkin math**
- raw KV payload is **not** the same as total deployment memory; real systems also need workspaces, allocator slack, communication buffers, page tables, and framework/runtime overhead

### General formula

For one active request, raw KV-cache bytes are:

```text
raw_kv_bytes_per_request
= layers × context_len × kv_heads × head_dim × 2(K and V) × bytes_per_element
```

For `B` active requests, multiply by `B`.

Important nuance:

- `kv_heads` means **number of key/value heads**
- for plain MHA, `kv_heads = num_attention_heads`
- for GQA/MQA, `kv_heads` is **smaller** than `num_attention_heads`
- if you accidentally use attention-head count instead of KV-head count on a GQA model, you will overestimate cache size

### Example A — MHA-style upper bound for scaling intuition

This example is useful for intuition, but it is **not** the actual KV-head configuration of Llama 2 70B.

Assume:
- layers = 80
- KV heads = 64
- head dim = 128
- context length = 4096
- batch size = 8 active requests
- dtype = fp16, so `bytes_per_element = 2`

Per-layer **K** payload:

```text
8 × 64 × 4096 × 128 × 2
= 536,870,912 bytes
= 536.87 MB (decimal)
= 512 MiB
```

Per-layer **V** payload is the same.

Per-layer **K+V** payload:

```text
1,073,741,824 bytes
= 1.07 GB (decimal)
= 1 GiB
```

Across all 80 layers:

```text
85,899,345,920 bytes
= 85.90 GB (decimal)
= 80 GiB
```

Per active request at 4K context in this MHA-style example:

```text
80 GiB / 8 = 10 GiB per request
```

### Example B — actual Llama 2 70B-style GQA accounting

Llama 2 70B uses grouped-query attention in the 70B model: it has 64 attention heads but only 8 KV heads.

Use the same assumptions as above, except:

- KV heads = 8

Per-layer **K** payload:

```text
8 × 8 × 4096 × 128 × 2
= 67,108,864 bytes
= 67.11 MB (decimal)
= 64 MiB
```

Per-layer **V** payload is the same.

Per-layer **K+V** payload:

```text
134,217,728 bytes
= 134.22 MB (decimal)
= 128 MiB
```

Across all 80 layers:

```text
10,737,418,240 bytes
= 10.74 GB (decimal)
= 10 GiB
```

Per active request at 4K context:

```text
10 GiB / 8 = 1.25 GiB per request
```

### Why this distinction matters

The MHA-style upper bound above is **8x larger** than the actual Llama 2 70B-style GQA accounting for the same batch, context length, and dtype.

That is exactly why senior engineers check:
- number of layers
- context length
- `num_key_value_heads`
- head dimension
- bytes per stored element
- concurrent active requests

before making capacity claims.

### Memory budgeting mental model for a real deployment

A cluster-level memory budget must include at least:

1. model weight payload across the cluster
2. active KV cache
3. temporary activations/workspaces during prefill
4. communication/runtime buffers
5. allocator and framework overhead

So a single one-line total like "deployment needs X GB" is usually too confident unless you also specify:
- model precision
- KV precision
- parallelism strategy
- interconnect / runtime
- scheduler caps
- prompt/output distributions

### What this means in a teaching environment

The weight footprint of a 70B model is usually the first blocker, not just the cache.

But KV math still matters because it scales linearly with:
- active requests
- context length
- number of layers
- KV heads
- head dimension
- bytes per stored element

That is the systems lesson: **cache size is architecture-dependent, workload-dependent, and concurrency-dependent.**

In [ ]:
# ============================================================
# Section 21 verification helper:
# exact KV-cache payload math for the assumptions in the notes.
#
# This cell is intentionally verbose because interview prep is
# high-stakes: we want the arithmetic to be easy to audit.
# ============================================================

# Helper that formats byte counts in both decimal and binary units.
# Decimal GB/MB are common in slides and product sheets.
# Binary GiB/MiB are common in systems work.
def format_bytes(n_bytes: int) -> str:
    gb_dec = n_bytes / 10**9
    mb_dec = n_bytes / 10**6
    gib = n_bytes / (2**30)
    mib = n_bytes / (2**20)
    return (
        f"{n_bytes:,} bytes"
        f" | {mb_dec:.2f} MB"
        f" | {gb_dec:.2f} GB"
        f" | {mib:.2f} MiB"
        f" | {gib:.2f} GiB"
    )

# Raw KV payload formula:
# layers * batch * context_len * kv_heads * head_dim
# * 2 (K and V) * bytes_per_element
def kv_cache_bytes(*, layers: int, batch: int, context_len: int, kv_heads: int, head_dim: int, bytes_per_element: int) -> int:
    return layers * batch * context_len * kv_heads * head_dim * 2 * bytes_per_element

layers = 80
batch = 8
context_len = 4096
head_dim = 128
bytes_per_element = 2  # fp16 / bf16 raw element size in bytes

# Case 1: MHA-style upper bound with 64 KV heads.
mha_total = kv_cache_bytes(
    layers=layers,
    batch=batch,
    context_len=context_len,
    kv_heads=64,
    head_dim=head_dim,
    bytes_per_element=bytes_per_element,
)

# Case 2: Llama-2-70B-style GQA with 8 KV heads.
gqa_total = kv_cache_bytes(
    layers=layers,
    batch=batch,
    context_len=context_len,
    kv_heads=8,
    head_dim=head_dim,
    bytes_per_element=bytes_per_element,
)

print("MHA-style upper bound (64 KV heads):")
print("  total raw K+V:", format_bytes(mha_total))
print("  per request   :", format_bytes(mha_total // batch))
print()

print("Llama-2-70B-style GQA (8 KV heads):")
print("  total raw K+V:", format_bytes(gqa_total))
print("  per request   :", format_bytes(gqa_total // batch))
print()

print("Reduction from 64 KV heads -> 8 KV heads:")
print(f"  {mha_total / gqa_total:.1f}x smaller raw KV payload")

## 22. Bottleneck Analysis: How Senior Engineers Approach Optimization

The biggest signal of a junior engineer is:
> "Let's optimize X."

The signal of a senior engineer is:
> "Let's measure the real bottleneck first."

### Important note on the numbers below

The examples in this section are **illustrative profile sketches**, not benchmark claims.
Use them to reason about where time may go.
Do **not** quote them as hardware- or model-specific measurements.

### The profiling mental model

Every inference workload has a hierarchy of potential bottlenecks:

```text
1. Scheduler / request queueing
   - Are requests arriving burstily?
   - Is the GPU underfed?
   - Are we wasting slots because the scheduler is too conservative?

2. Prefill efficiency
   - Is prompt processing compute-heavy enough to benefit from bigger batches?
   - Are long prompts causing queue buildup?

3. Decode efficiency
   - Is per-token generation memory-bandwidth limited?
   - Are we spending more time moving state than doing useful math?

4. Communication / network
   - Are collective operations dominating?
   - Is cross-GPU traffic erasing parallelism gains?
```

### How to profile with PyTorch

```python
import torch.profiler as profiler

with profiler.profile(
    activities=[profiler.ProfilerActivity.CPU, profiler.ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    model.generate_with_cache(prompt, max_new_tokens=8)

prof.key_averages().table(sort_by="cuda_time_total", row_limit=10)
```

What to look for:
- where wall-clock time actually goes
- whether the GPU is compute-bound or memory-bound
- whether the front-end/scheduler is underfeeding the device
- whether communication or allocator behavior shows up as a hidden tax

### One plausible profile sketch for prefill (illustrative only)

| Component | Rough share | Interpretation |
|----------|-------------|----------------|
| MLP / projections | large | can be a major compute consumer |
| Attention projections + score/value path | large | often important, but not always dominant |
| Softmax / normalization / elementwise work | moderate | usually matters less than the big matmuls |
| Misc. runtime overhead | small-to-moderate | launch, framework, allocator, bookkeeping |

### One plausible profile sketch for decode (illustrative only)

| Component | Rough share | Interpretation |
|----------|-------------|----------------|
| Reading weights + KV state | often dominant | memory movement can be the ceiling |
| Attention math for the new token | smaller than people expect | compute may no longer dominate |
| Framework / scheduler / dispatch overhead | visible at small batch sizes | easy to underestimate |
| Communication | can dominate in distributed decode | especially when TP degree is high |

### Decision framework: "What should I optimize?"

Ask in order:

1. **Is the GPU clearly underutilized?**
   - If yes, look at batching, queueing, tokenization, router logic, and scheduler policy first.

2. **Is prefill the dominant wall-clock cost?**
   - If yes, larger prompt batches, better kernels, and better prompt scheduling may help.

3. **Is decode the dominant wall-clock cost?**
   - If yes, think memory system, KV footprint, batching policy, and communication overhead before writing a custom kernel.

4. **Is distributed communication dominating?**
   - If yes, revisit TP degree, placement, and whether the workload mix actually benefits from that parallelism choice.

### The uncomfortable truth

In many production systems, end-to-end wins come more often from:
- better scheduling
- better batching
- better memory management
- better admission control

than from a single isolated kernel tweak.

Kernel work still matters, but only after you confirm it sits on the critical path.

## 23. Continuous Batching: Why Scheduling Beats Kernels

This is where many real serving systems get large practical wins.

### Important note on the timeline below

The times below are **toy napkin math** meant to illustrate queueing behavior.
They are **not benchmark numbers**.

### Static batching (the old way)

```text
Time 0.0: Request 1 arrives (long prompt, then long decode)
Time 0.5: Request 2 arrives but waits for a compatible batch slot
Time 2.0: Request 1 finishes prefill and starts decode
Time 2.0+: Request 2 finally begins service
...
Outcome: visible idle gaps and poorer latency for late arrivals
```

### Continuous batching (modern serving mental model)

```text
Time 0.0: Request 1 prefills
Time 0.5: Request 1 decodes while Request 2 can start prefill
Time 1.0: Request 1 keeps decoding, Request 2 moves forward, Request 3 can join
...
Outcome: fewer idle gaps and better overlap between different request phases
```

### Why this works

Key idea: **prefill and decode have different resource profiles**.

- Prefill tends to have more dense prompt-side work
- Decode tends to be more state- and memory-heavy per generated token

A scheduler that mixes them intelligently can keep the device busier than a simple "wait for a static batch, then run it" policy.

### A better way to talk about impact

Instead of quoting one magic throughput number, think in terms of:

- **utilization**: fewer idle gaps
- **throughput**: more useful tokens or requests completed per unit time
- **latency balance**: less waiting for newly arrived interactive requests
- **memory pressure**: more active requests means more KV state, so scheduler policy still has to respect capacity

### Illustrative qualitative comparison

| Approach | Utilization tendency | Latency tendency | Notes |
|----------|----------------------|------------------|-------|
| Small static batches | lower | can be okay at low load | simple but leaves performance on the table |
| Large static batches | somewhat higher | can be worse because of waiting | better throughput, worse queueing behavior |
| Continuous batching | often best balance | often better for mixed workloads | needs stronger scheduling + memory management |

### The scheduler's real job

A production scheduler has to do more than "make batches":

1. track per-request state
2. decide which requests to admit
3. reserve enough KV capacity
4. avoid starving short interactive requests behind very long jobs
5. reclaim memory quickly when requests finish
6. keep the device busy without blowing up p95 latency

### Interview signal

A good infrastructure answer sounds like this:

> "Continuous batching is a scheduling win first. It usually improves utilization because it reduces idle gaps and overlaps different request phases, but the exact gain depends on prompt lengths, output lengths, admission caps, and KV capacity."

That is a stronger answer than quoting one fixed throughput multiplier with no workload context.

## 24. Production KV Cache Techniques: What Actually Gets Deployed

The notebook uses a naive dense KV cache. Real serving systems are more careful.

### Important note on the numbers below

- when I give raw cache sizes, they are exact arithmetic for the stated tensor shapes
- when I talk about latency or quality impact, treat it as architectural intuition rather than a benchmark claim
- quantized-cache numbers below are **raw payload math** unless I explicitly say otherwise; real implementations also carry scales, packing overhead, allocator state, and metadata

### Problem 1: Memory fragmentation

Naive dense approach:

```text
Request 1 allocates one large contiguous cache region
Request 2 allocates another
Request 1 finishes
Now free space exists, but not necessarily in the right contiguous shape
```

That creates classic allocator pain: **external fragmentation**.

### Solution: Paged / blocked KV allocation

Instead of one giant contiguous tensor, blocked runtimes split KV state into fixed-size pieces.

A useful mental model is a block size of **16 tokens** with `head_dim = 128`.

For **one KV head** at fp16:

```text
K block payload  = 16 × 128 × 2 bytes = 4,096 bytes  = 4 KiB
V block payload  = 16 × 128 × 2 bytes = 4,096 bytes  = 4 KiB
K+V block payload per head            = 8,192 bytes  = 8 KiB
```

Scale that up:

- across **64 KV heads** in one MHA layer:
  - `16 × 64 × 128 × 2(K,V) × 2 bytes = 524,288 bytes = 512 KiB`
- across **80 layers** for one request:
  - `512 KiB × 80 = 40 MiB`

For an actual **Llama 2 70B-style GQA** layout with 8 KV heads:

- one layer, one 16-token increment:
  - `16 × 8 × 128 × 2(K,V) × 2 bytes = 65,536 bytes = 64 KiB`
- across 80 layers for one request:
  - `64 KiB × 80 = 5 MiB`

The exact implementation details vary by runtime, but the scaling idea is the same:
small fixed-size blocks make allocation, reuse, and eviction much easier.

### Problem 2: KV cache can dominate memory budgets

Using the same 80-layer, 4K-context, batch-8, fp16 example:

| Attention style | KV heads | Total raw K+V payload |
|-----------------|----------|------------------------|
| MHA | 64 | 80 GiB |
| GQA | 8 | 10 GiB |
| MQA | 1 | 1.25 GiB |

So architecture choice alone changes KV size dramatically.

### MQA and GQA

Standard MHA:

```text
num_attention_heads = num_kv_heads
cache scales with all heads
```

Grouped-query attention (GQA):

```text
num_attention_heads > num_kv_heads
multiple query heads share each KV head group
```

Multi-query attention (MQA):

```text
num_kv_heads = 1
all query heads share one KV head
```

**Exact cache reduction relative to MHA in this example:**
- GQA with 8 KV heads: `64 / 8 = 8x` smaller
- MQA with 1 KV head: `64 / 1 = 64x` smaller

### Problem 3: Quantizing the cache

If we hold the tensor shape fixed and only change stored element size:

For the **GQA example above** (`10 GiB` raw K+V at fp16/bf16):

- 2 bytes/element (fp16/bf16): `10 GiB`
- 1 byte/element (fp8/int8-style raw payload): `5 GiB`
- 4 bits/element theoretical payload: `2.5 GiB` before scales/packing overhead

For the **MHA upper-bound example** (`80 GiB` raw K+V at fp16/bf16):

- 2 bytes/element: `80 GiB`
- 1 byte/element: `40 GiB`
- 4 bits/element theoretical payload: `20 GiB` before scales/packing overhead

So quantization helps, but the exact real memory win depends on implementation details.

### What production systems actually layer on top

A serious serving stack usually combines several ideas:

1. blocked / paged KV allocation
2. exact-prefix reuse when token prefixes match
3. architecture-aware KV sizing (MHA vs GQA vs MQA)
4. optional KV quantization and/or offloading
5. eviction / admission control when active cache pressure is high

### Interview signal

A strong answer is not:

> "Use paged attention and int8."

A stronger answer is:

> "First compute the raw KV payload from layers, context, KV heads, head dimension, batch, and bytes per element. Then decide whether the better lever is model architecture, quantization, paging, reuse, or admission control."

That shows you are reasoning from first principles instead of memorizing buzzwords.

## 24A. PagedAttention and vLLM: from dense KV cache to paged block tables

This section adds a more concrete mental model for **PagedAttention** and for how **vLLM** uses paged KV storage in a serving system.

### What this section is trying to teach

1. why a **dense contiguous KV cache** creates allocator pain  
2. what it means to split KV state into **fixed-size logical blocks**  
3. how a **block table** maps a sequence's logical blocks to non-contiguous physical blocks  
4. why prefix sharing becomes natural with **reference counts** and **copy-on-write**  
5. where that paged-KV idea fits into a simplified **vLLM serving architecture**

### What this section is **not**

This is **not** a reimplementation of the real vLLM kernel.

Important simplifications:

- Real vLLM stores K/V tensors with much richer layout details than this notebook shows.
- In the real PagedAttention design, each block stores a fixed number of tokens **at one head**.  
- The toy allocator below collapses those lower-level tensor details into an **abstract block** so the mapping logic is easier to see.
- The architecture walkthrough is a **high-level mental model** based on the vLLM V1 docs, not a line-by-line code tour.

That is exactly the right level for interview preparation: understand the **resource model** and the **scheduler / memory interaction**, not every kernel template argument.

In [ ]:
# ============================================================
# Dense contiguous KV cache vs paged/block-based KV cache
# ------------------------------------------------------------
# This cell draws two educational diagrams:
#   1) why naive contiguous allocation can fragment memory
#   2) how a block table lets a sequence use non-contiguous
#      physical blocks while preserving logical token order
#
# The goal is not to match every real vLLM internal data
# structure. The goal is to make the interview mental model
# visually obvious.
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

# ------------------------------------------------------------
# Helper: draw a stack of memory "slots" so we can reason about
# fragmentation in a simple, OS-like way.
# ------------------------------------------------------------
def draw_memory_stack(ax, title, total_slots, occupied, annotations=None):
    """
    Draw a vertical memory stack.

    Parameters
    ----------
    ax : matplotlib axis
        Axis to draw on.
    title : str
        Title for the subplot.
    total_slots : int
        Total number of physical slots in this toy memory.
    occupied : dict[int, tuple[str, str]]
        Mapping slot_index -> (label, facecolor).
    annotations : list[str] | None
        Optional extra notes drawn below the picture.
    """
    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlim(0, 3.8)
    ax.set_ylim(-3.2, total_slots + 0.5)
    ax.axis("off")

    # Draw memory from bottom to top so the picture feels like
    # a memory address range.
    for slot in range(total_slots):
        y = slot
        label, color = occupied.get(slot, ("FREE", "#f2f2f2"))
        rect = Rectangle((0.5, y), 1.0, 0.9, linewidth=1.2, edgecolor="black", facecolor=color)
        ax.add_patch(rect)
        ax.text(1.0, y + 0.45, label, ha="center", va="center", fontsize=9)
        ax.text(0.25, y + 0.45, f"{slot}", ha="center", va="center", fontsize=8)

    ax.text(1.0, total_slots + 0.15, "physical memory slots", ha="center", va="bottom", fontsize=9)

    if annotations:
        for i, note in enumerate(annotations):
            ax.text(2.0, -0.4 - i * 0.55, note, ha="left", va="top", fontsize=9)


# ------------------------------------------------------------
# Left panel: dense contiguous allocation.
#
# We show a situation where there is enough total free memory,
# but not enough *contiguous* free memory for a new request.
# This is the classic fragmentation story.
# ------------------------------------------------------------
dense_occupied = {
    0: ("A", "#cfe2f3"),
    1: ("A", "#cfe2f3"),
    2: ("A", "#cfe2f3"),
    3: ("A", "#cfe2f3"),
    4: ("FREE", "#f2f2f2"),  # a hole left behind by a completed request
    5: ("FREE", "#f2f2f2"),
    6: ("C", "#d9ead3"),
    7: ("C", "#d9ead3"),
    8: ("C", "#d9ead3"),
    9: ("FREE", "#f2f2f2"),
    10: ("FREE", "#f2f2f2"),
    11: ("FREE", "#f2f2f2"),
}

# ------------------------------------------------------------
# Right panel: paged/block-based allocation.
#
# Here, sequence D needs 4 logical blocks. Those logical blocks
# do NOT need to be physically adjacent. The block table can
# map them to scattered physical blocks.
#
# We also show another sequence E sharing D's first two logical
# blocks, which is the block-table mental model behind prefix
# sharing.
# ------------------------------------------------------------
def draw_paged_mapping(ax):
    ax.set_title("PagedAttention mental model", fontsize=12, weight="bold")
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 11)
    ax.axis("off")

    # Logical blocks for sequence D.
    logical_y_d = 8.2
    ax.text(0.4, logical_y_d + 0.55, "Seq D logical blocks", fontsize=10, weight="bold")
    d_positions = []
    for i in range(4):
        x = 0.7 + i * 1.4
        d_positions.append((x, logical_y_d))
        rect = Rectangle((x, logical_y_d), 1.0, 0.7, edgecolor="black", facecolor="#fff2cc")
        ax.add_patch(rect)
        ax.text(x + 0.5, logical_y_d + 0.35, f"L{i}", ha="center", va="center", fontsize=9)

    # Logical blocks for sequence E: shares the first two blocks,
    # then diverges. This is an educational picture of prefix reuse.
    logical_y_e = 6.5
    ax.text(0.4, logical_y_e + 0.55, "Seq E logical blocks", fontsize=10, weight="bold")
    e_positions = []
    for i in range(3):
        x = 0.7 + i * 1.4
        e_positions.append((x, logical_y_e))
        rect = Rectangle((x, logical_y_e), 1.0, 0.7, edgecolor="black", facecolor="#d9ead3")
        ax.add_patch(rect)
        ax.text(x + 0.5, logical_y_e + 0.35, f"L{i}", ha="center", va="center", fontsize=9)

    # Physical blocks are deliberately scattered, not contiguous.
    ax.text(8.2, 9.3, "Physical blocks in GPU memory", fontsize=10, weight="bold")
    physical_positions = {
        1: (8.7, 8.0),
        6: (10.2, 7.0),
        2: (8.7, 6.0),
        10: (10.2, 5.0),
        11: (8.7, 4.0),
    }
    for block_id, (x, y) in physical_positions.items():
        rect = Rectangle((x, y), 1.2, 0.8, edgecolor="black", facecolor="#ead1dc")
        ax.add_patch(rect)
        ax.text(x + 0.6, y + 0.4, f"P{block_id}", ha="center", va="center", fontsize=9)

    # Block table mapping for D:
    #   L0 -> P1
    #   L1 -> P6
    #   L2 -> P2
    #   L3 -> P10
    mapping_d = [1, 6, 2, 10]
    for i, phys in enumerate(mapping_d):
        lx, ly = d_positions[i]
        px, py = physical_positions[phys]
        arrow = FancyArrowPatch(
            (lx + 1.0, ly + 0.35),
            (px, py + 0.4),
            arrowstyle="->",
            mutation_scale=12,
            linewidth=1.2,
        )
        ax.add_patch(arrow)

    # Block table mapping for E:
    #   L0 -> P1   (shared prefix)
    #   L1 -> P6   (shared prefix)
    #   L2 -> P11  (diverged continuation)
    mapping_e = [1, 6, 11]
    for i, phys in enumerate(mapping_e):
        lx, ly = e_positions[i]
        px, py = physical_positions[phys]
        arrow = FancyArrowPatch(
            (lx + 1.0, ly + 0.35),
            (px, py + 0.4),
            arrowstyle="->",
            mutation_scale=12,
            linewidth=1.2,
            linestyle="--",
        )
        ax.add_patch(arrow)

    ax.text(
        0.4,
        3.2,
        "Key idea:\nlogical token order stays contiguous for the sequence,\n"
        "but physical KV blocks can live anywhere in memory.",
        fontsize=9,
        va="top",
    )
    ax.text(
        8.0,
        2.0,
        "Dashed arrows = shared prefix blocks\n"
        "A block table + refcounts make sharing practical.",
        fontsize=9,
        va="top",
    )


fig, axes = plt.subplots(1, 2, figsize=(15, 6))

draw_memory_stack(
    axes[0],
    title="Naive dense contiguous allocation",
    total_slots=12,
    occupied=dense_occupied,
    annotations=[
        "Request D needs 4 blocks.",
        "Total free blocks = 5, but largest contiguous run = 3.",
        "Enough memory in total, but not in one contiguous chunk.",
    ],
)

draw_paged_mapping(axes[1])

plt.tight_layout()
plt.show()

### What the diagrams mean

The left diagram is the allocator problem with a naive dense cache:

- each request wants one large contiguous region  
- when requests finish at different times, holes appear  
- a new request may fail to find one large contiguous region even if total free space is sufficient  

The right diagram is the PagedAttention mental model:

- break a sequence's KV state into **logical blocks**
- map each logical block to a **physical block** using a **block table**
- physical blocks do **not** need to be adjacent in memory
- a second sequence can share the same physical prefix blocks by pointing to them in its own block table

That is why people compare PagedAttention to a virtual-memory idea:

- **sequence** ~ process  
- **logical KV blocks** ~ virtual pages  
- **physical KV blocks** ~ physical frames  
- **block table** ~ page table  

One more precision note:

- In the real vLLM kernel, a block stores a fixed number of tokens **for one head**
- In this notebook, one "block" is an **educational abstraction** that stands in for the larger KV payload you would actually manage across layers / heads / tensors

In [ ]:
# ============================================================
# Toy block allocator with block tables, prefix sharing,
# reference counts, and copy-on-write (COW)
# ------------------------------------------------------------
# This allocator is deliberately educational:
#   - one "block" is an abstract teaching block
#   - each sequence has a block table: logical -> physical
#   - physical blocks can be shared by multiple sequences
#   - if a shared partial block would be modified, we perform
#     a copy-on-write step first
#
# This is NOT the real vLLM implementation. It is a compact
# mental model that makes interview explanations much easier.
# ============================================================

from dataclasses import dataclass, field
from collections import deque
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

@dataclass
class ToyPhysicalBlock:
    """
    A tiny educational representation of one physical KV block.

    Real systems store tensor payloads, not strings.
    We store token labels so the block-table behavior is easy to see.
    """
    block_id: int
    capacity: int
    tokens: list = field(default_factory=list)
    refcount: int = 0

    @property
    def used(self):
        return len(self.tokens)

    @property
    def is_full(self):
        return self.used >= self.capacity


class ToyPagedKVAllocator:
    """
    Minimal paged-KV allocator for interview intuition.

    Main ideas:
    - free_list: which physical blocks are currently available
    - sequences[seq_id]: block table for one sequence
    - blocks[block_id]: physical block state + refcount
    """
    def __init__(self, num_blocks=12, block_size=4):
        self.block_size = block_size
        self.blocks = {
            block_id: ToyPhysicalBlock(block_id=block_id, capacity=block_size)
            for block_id in range(num_blocks)
        }
        self.free_list = deque(range(num_blocks))
        self.sequences = {}

    def _allocate_empty_block(self):
        """Allocate one unused physical block."""
        if not self.free_list:
            raise RuntimeError("Out of physical blocks in the toy allocator.")
        block_id = self.free_list.popleft()
        block = self.blocks[block_id]
        block.tokens = []
        block.refcount = 1
        return block_id

    def _clone_block_for_cow(self, old_block_id):
        """
        Copy-on-write:
        allocate a new block and copy the old token labels into it.

        We do this only when a sequence wants to modify a shared,
        not-yet-full block.
        """
        old_block = self.blocks[old_block_id]
        new_block_id = self._allocate_empty_block()
        new_block = self.blocks[new_block_id]
        new_block.tokens = list(old_block.tokens)
        return new_block_id

    def create_sequence(self, seq_id, token_labels):
        """Create a brand-new sequence from scratch."""
        if seq_id in self.sequences:
            raise ValueError(f"Sequence {seq_id} already exists.")
        self.sequences[seq_id] = []

        # Append token-by-token so the same logic is used everywhere.
        for tok in token_labels:
            self.append_token(seq_id, tok)

    def fork_sequence(self, parent_seq_id, child_seq_id):
        """
        Create a child sequence that initially shares all parent's blocks.

        This is the educational prefix-sharing story used in beam search
        or parallel sampling discussions.
        """
        if child_seq_id in self.sequences:
            raise ValueError(f"Sequence {child_seq_id} already exists.")
        parent_table = self.sequences[parent_seq_id]
        self.sequences[child_seq_id] = list(parent_table)

        # Every shared physical block gets one more reference.
        for block_id in parent_table:
            self.blocks[block_id].refcount += 1

    def append_token(self, seq_id, token_label):
        """
        Append one token to a sequence.

        Cases:
        1) no blocks yet -> allocate first block
        2) last block full -> allocate a new block
        3) last block shared and partial -> copy-on-write, then append
        4) last block unique and partial -> append in place
        """
        table = self.sequences[seq_id]

        if not table:
            # First token for this sequence.
            new_block_id = self._allocate_empty_block()
            table.append(new_block_id)

        last_block_id = table[-1]
        last_block = self.blocks[last_block_id]

        if last_block.is_full:
            # Need a fresh block because the current one is full.
            new_block_id = self._allocate_empty_block()
            table.append(new_block_id)
            last_block_id = new_block_id
            last_block = self.blocks[last_block_id]

        elif last_block.refcount > 1:
            # Shared partial block: modifying it would affect another
            # sequence, so we first create a private copy.
            new_block_id = self._clone_block_for_cow(last_block_id)
            self.blocks[last_block_id].refcount -= 1
            table[-1] = new_block_id
            last_block_id = new_block_id
            last_block = self.blocks[last_block_id]

        # Safe to append now.
        last_block.tokens.append(token_label)

    def append_tokens(self, seq_id, token_labels):
        """Convenience helper for multiple appends."""
        for tok in token_labels:
            self.append_token(seq_id, tok)

    def free_sequence(self, seq_id):
        """
        Release a sequence.

        Physical blocks return to the free list only when refcount
        reaches zero.
        """
        table = self.sequences.pop(seq_id)
        for block_id in table:
            block = self.blocks[block_id]
            block.refcount -= 1
            if block.refcount == 0:
                block.tokens = []
                self.free_list.append(block_id)

    def sequence_table(self, seq_id):
        """Return a readable logical->physical summary."""
        table = self.sequences[seq_id]
        rows = []
        for logical_idx, block_id in enumerate(table):
            block = self.blocks[block_id]
            rows.append(
                {
                    "seq": seq_id,
                    "logical_block": logical_idx,
                    "physical_block": block_id,
                    "used_tokens": block.used,
                    "refcount": block.refcount,
                    "tokens": " ".join(block.tokens),
                }
            )
        return rows

    def print_state(self, title):
        """Pretty-print the current allocator state."""
        print("=" * 72)
        print(title)
        print("=" * 72)
        print(f"block_size = {self.block_size} tokens per block")
        print(f"free physical blocks = {list(self.free_list)}")
        print()

        for seq_id in sorted(self.sequences):
            print(f"Sequence: {seq_id}")
            for row in self.sequence_table(seq_id):
                print(
                    f"  L{row['logical_block']} -> P{row['physical_block']} | "
                    f"used={row['used_tokens']}/{self.block_size} | "
                    f"refcount={row['refcount']} | tokens=[{row['tokens']}]"
                )
            print()

        print("Physical block summary:")
        for block_id in sorted(self.blocks):
            block = self.blocks[block_id]
            status = "FREE" if block.refcount == 0 else "IN-USE"
            print(
                f"  P{block_id}: {status:6s} | refcount={block.refcount} | "
                f"used={block.used}/{block.capacity} | tokens={block.tokens}"
            )
        print()

    def draw_state(self, title):
        """
        Draw the current block-table mapping.

        Left side  = per-sequence logical blocks
        Right side = physical block pool with refcounts
        """
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.set_title(title, fontsize=12, weight="bold")
        ax.set_xlim(0, 15)
        ax.set_ylim(0, 14)
        ax.axis("off")

        # Draw sequences and their logical blocks.
        seq_ids = sorted(self.sequences)
        seq_y_positions = {}
        for seq_idx, seq_id in enumerate(seq_ids):
            base_y = 12.5 - seq_idx * 2.4
            seq_y_positions[seq_id] = base_y
            ax.text(0.5, base_y + 0.4, seq_id, fontsize=10, weight="bold", va="center")

            for logical_idx, block_id in enumerate(self.sequences[seq_id]):
                x = 2.0 + logical_idx * 1.3
                rect = Rectangle((x, base_y), 1.0, 0.7, edgecolor="black", facecolor="#fff2cc")
                ax.add_patch(rect)
                ax.text(x + 0.5, base_y + 0.35, f"L{logical_idx}", ha="center", va="center", fontsize=9)

                # Arrow from logical block to physical block.
                phys_x = 9.5
                phys_y = 12.5 - block_id * 0.9
                arrow = FancyArrowPatch(
                    (x + 1.0, base_y + 0.35),
                    (phys_x, phys_y + 0.25),
                    arrowstyle="->",
                    mutation_scale=10,
                    linewidth=1.0,
                )
                ax.add_patch(arrow)

        # Draw physical block pool on the right.
        ax.text(9.5, 13.5, "physical blocks", fontsize=10, weight="bold")
        for block_id in sorted(self.blocks):
            block = self.blocks[block_id]
            x = 9.5
            y = 12.5 - block_id * 0.9
            color = "#ead1dc" if block.refcount > 0 else "#f2f2f2"
            rect = Rectangle((x, y), 3.8, 0.55, edgecolor="black", facecolor=color)
            ax.add_patch(rect)

            token_text = " ".join(block.tokens) if block.tokens else "(free)"
            ax.text(
                x + 0.1,
                y + 0.275,
                f"P{block_id} | ref={block.refcount} | used={block.used}/{block.capacity} | {token_text}",
                ha="left",
                va="center",
                fontsize=8,
            )

        plt.tight_layout()
        plt.show()


# ------------------------------------------------------------
# Demonstration scenario
# ------------------------------------------------------------
# 1) Create a prompt sequence with 6 tokens.
#    With block_size=4, that becomes:
#      - one full block for A B C D
#      - one partial block for E F
#
# 2) Fork two child sequences that share the prompt blocks.
#
# 3) Append to the children.
#    Because the last shared block is only partially full,
#    each child needs a copy-on-write step before modifying it.
#
# 4) Free one child and show how refcounts change.
# ------------------------------------------------------------
allocator = ToyPagedKVAllocator(num_blocks=12, block_size=4)

allocator.create_sequence("prompt", list("ABCDEF"))
allocator.print_state("State 1: prompt created")

allocator.fork_sequence("prompt", "beam_1")
allocator.fork_sequence("prompt", "beam_2")
allocator.print_state("State 2: two child sequences share the prompt blocks")

# beam_1 appends one token.
# Its last block was shared and partial, so this triggers COW.
allocator.append_tokens("beam_1", ["g"])

# beam_2 appends three tokens.
# It also starts from the same shared partial block, so it gets
# its own copy first, then fills it, then continues if needed.
allocator.append_tokens("beam_2", ["x", "y", "z"])

allocator.print_state("State 3: children diverge; shared partial block triggers COW")
allocator.draw_state("State 3: after divergence and copy-on-write")

allocator.free_sequence("beam_1")
allocator.print_state("State 4: beam_1 freed; blocks return only when refcount hits zero")
allocator.draw_state("Final allocator view: after divergence, COW, and freeing beam_1")

### Short vLLM architecture walkthrough

Here is the **high-level serving mental model** to carry into interviews.

#### 1. API server process
The front-end process handles things like:

- HTTP / OpenAI-compatible requests
- input processing and tokenization
- streaming responses back to clients

#### 2. Engine core process
This is the part you should associate with **serving intelligence**:

- scheduler
- KV-cache management
- deciding which requests advance in each step
- dispatching work to the GPU workers

This is also the natural place to think about **block tables**, **free blocks**, **sharing**, and **admission pressure** in a high-level design discussion.

#### 3. GPU worker processes
Each GPU worker:

- owns a GPU
- loads model weights
- executes forward passes
- manages GPU memory for the work it runs

In a tensor-parallel or pipeline-parallel deployment, there are multiple workers.

#### 4. LLMEngine / AsyncLLMEngine mental model
A compact way to explain vLLM in an interview is:

> "The serving stack has a front-end request layer, an engine that handles scheduling and KV-cache state, and GPU workers that execute the model. PagedAttention makes the KV side memory-efficient enough to sustain better batching and sharing."

That is not the whole codebase, but it is the **right architecture story**.

### Important precision note

The pictures below are **teaching diagrams**, not exact reproductions of the official vLLM architecture figures. They are intentionally simplified so the control-plane story is easy to memorize.

In [ ]:
# ============================================================
# Simplified vLLM architecture walkthrough diagram
# ------------------------------------------------------------
# This is a teaching diagram, not an exact official figure.
# It is designed to help you explain the system in an interview.
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 15)
ax.set_ylim(0, 8)
ax.axis("off")
ax.set_title("Simplified vLLM mental model for interviews", fontsize=13, weight="bold")

# API server box
api_box = Rectangle((0.6, 2.2), 3.0, 3.2, edgecolor="black", facecolor="#cfe2f3", linewidth=1.5)
ax.add_patch(api_box)
ax.text(2.1, 4.9, "API server", ha="center", va="center", fontsize=11, weight="bold")
ax.text(
    2.1,
    3.6,
    "HTTP / OpenAI API\ninput processing\nstreaming responses",
    ha="center",
    va="center",
    fontsize=9,
)

# Engine core box
engine_box = Rectangle((5.0, 1.6), 4.0, 4.4, edgecolor="black", facecolor="#fff2cc", linewidth=1.5)
ax.add_patch(engine_box)
ax.text(7.0, 5.4, "Engine core", ha="center", va="center", fontsize=11, weight="bold")
ax.text(
    7.0,
    3.9,
    "scheduler\nKV-cache manager\nrequest selection\nwork dispatch",
    ha="center",
    va="center",
    fontsize=9,
)
ax.text(
    7.0,
    2.2,
    "Think here about:\nblock tables, free blocks,\nsharing, admission pressure",
    ha="center",
    va="center",
    fontsize=8,
)

# GPU worker boxes
worker_positions = [(10.6, 4.2), (10.6, 2.4)]
for i, (x, y) in enumerate(worker_positions):
    box = Rectangle((x, y), 3.4, 1.3, edgecolor="black", facecolor="#d9ead3", linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + 1.7, y + 0.9, f"GPU worker {i}", ha="center", va="center", fontsize=10, weight="bold")
    ax.text(x + 1.7, y + 0.45, "weights + forward pass\nGPU memory ownership", ha="center", va="center", fontsize=8)

# Optional DP coordinator box, shown lightly because it is conditional.
dp_box = Rectangle((10.6, 0.5), 3.4, 1.0, edgecolor="black", facecolor="#ead1dc", linewidth=1.0, linestyle="--")
ax.add_patch(dp_box)
ax.text(12.3, 1.0, "DP coordinator (optional)", ha="center", va="center", fontsize=9)

# Arrows between components.
arrow_1 = FancyArrowPatch((3.6, 3.8), (5.0, 3.8), arrowstyle="->", mutation_scale=14, linewidth=1.5)
arrow_2 = FancyArrowPatch((9.0, 4.3), (10.6, 4.85), arrowstyle="->", mutation_scale=14, linewidth=1.5)
arrow_3 = FancyArrowPatch((9.0, 3.3), (10.6, 3.05), arrowstyle="->", mutation_scale=14, linewidth=1.5)
ax.add_patch(arrow_1)
ax.add_patch(arrow_2)
ax.add_patch(arrow_3)

# Client and response annotations.
ax.text(0.2, 6.8, "clients", fontsize=10, weight="bold")
ax.text(0.2, 6.1, "chat app\nagent\nbatch job", fontsize=8, va="top")
client_arrow = FancyArrowPatch((0.8, 5.8), (1.7, 5.3), arrowstyle="->", mutation_scale=12, linewidth=1.2)
resp_arrow = FancyArrowPatch((1.7, 2.3), (0.8, 1.6), arrowstyle="->", mutation_scale=12, linewidth=1.2)
ax.add_patch(client_arrow)
ax.add_patch(resp_arrow)
ax.text(0.25, 1.0, "streamed tokens\nback to client", fontsize=8)

plt.tight_layout()
plt.show()

## 25. Distributed Inference: Multi-GPU Strategies

When a model does not fit comfortably on one GPU, you need parallelism.  
The hard part is that communication is real work too.

### Important note on the numbers below

This section uses **toy napkin math** to reason about compute-vs-communication trade-offs.
These are **not benchmark numbers**.

### Setup

A 70B fp16 model has roughly:

```text
70B parameters × 2 bytes ≈ 140 GB of raw weight payload
```

So without aggressive quantization, multi-GPU placement is usually required.

### Strategy 1: Tensor Parallelism (split weights within a layer)

Very rough mental model:

```text
single_gpu_time   ≈ local_compute
tp_time           ≈ (local_compute / tp_degree) + collective_overhead
```

Toy example for **prefill-like** work:

- suppose one layer group would take `18 ms` of local compute on one GPU
- suppose collective overhead is `5 ms`

Then:

- 2-way TP: `18/2 + 5 = 14 ms`
- 4-way TP: `18/4 + 5 = 9.5 ms`
- 8-way TP: `18/8 + 5 = 7.25 ms`

This is why TP can help when local compute is still large:
the collective cost hurts, but the compute term is big enough that splitting work still helps.

### Why decode is trickier

For **decode**, the local compute per generated token can be much smaller, while collective overhead does not disappear.

Toy example:

- local compute per token on one GPU: `4 ms`
- collective overhead: `5 ms`

Then a high-TP decode path can be worse than expected because:

```text
tp_decode_time ≈ (4 / tp_degree) + 5
```

At high TP degree, the `+ 5` term dominates.
That is the systems intuition behind the statement:
**distributed decode often scales much worse than distributed prefill.**

### Strategy 2: Sequence Parallelism

Sequence parallelism splits work across token positions.

That can help certain workloads, but attention often still needs information from the full effective context, so communication remains important.
It is not a free lunch.

### Strategy 3: Pipeline Parallelism

Pipeline parallelism splits layers across GPUs:

```text
GPU0: early layers
GPU1: next layers
...
```

It helps memory fitting, but decode with tiny microbatches can suffer from **pipeline bubbles**:
many stages spend time waiting instead of doing useful work.

### Practical engineering mental model

- **Prefill** often benefits more from broader parallelism because there is more prompt-side compute to amortize communication.
- **Decode** is often more sensitive to communication overhead and KV-state movement.
- The best setup depends on:
  - interconnect quality
  - TP degree
  - prompt/output length distribution
  - latency target vs throughput target
  - whether the service is interactive or offline/batch

### Safer interview phrasing

A strong answer sounds like this:

> "Tensor parallelism often helps prefill more than decode because prefill has more local compute to amortize collectives. Decode can become communication-dominated at high TP degree, so I would validate the TP setting against the actual workload instead of assuming more GPUs automatically means lower latency."

That is better than quoting one hard latency number with no hardware context.

## 26. Real Infrastructure Interview Questions with Weak/Strong Answers

These questions separate people who understand inference systems from people who only know model buzzwords.

### Important note on the numbers below

Where I use arithmetic below, it is **back-of-the-envelope reasoning**.
It is there to structure the answer, not to serve as a benchmark claim.

### Question 1: "We optimized the attention kernel 4x. But throughput only improved 1.2x. Why?"

**Weak answer:**
> "Maybe there's a bug in the kernel? Or we need to optimize other operations too?"

**Problem:** It guesses instead of locating the bottleneck.

---

**Strong answer:**
> "I would first ask whether the system is prefill-bound or decode-bound.
>
> If decode dominates, a much faster attention kernel may not move end-to-end throughput very much because decode can be limited by memory movement, KV access, scheduling, or communication.
>
> If prefill dominates, then the kernel still might not be the whole story. Suppose attention was only about a quarter of end-to-end time. A 4x speedup on that quarter would not yield a 4x system speedup.
>
> So my next step is profiling: wall-clock breakdown, GPU utilization, memory bandwidth pressure, and scheduler behavior."

**Why it's strong:** It reasons from Amdahl's law and the real system bottleneck.

---

### Question 2: "Design a serving system for 100k tokens/day on a single GPU."

**Weak answer:**
> "Use a batch size of 32. Process requests in order. Should be fine."

**Problem:** It treats average load as the whole problem.

---

**Strong answer:**
> "First I would normalize the load:
> - `100,000 / 86,400 ≈ 1.16` tokens/sec on average
>
> So average load is modest. The real design question is burstiness and SLOs, not the daily average.
>
> I would ask:
> - how bursty is traffic?
> - what are prompt and output length distributions?
> - what p95 TTFT / latency do we need?
>
> Then I would choose a simple serving stack:
> - small admission queue
> - continuous batching if bursts matter
> - hard caps on max prompt/output length
> - KV-aware concurrency limits
>
> If the average really is that low, I would also question whether a smaller model or cheaper deployment tier is the better business choice."

**Why it's strong:** It starts with arithmetic, then pivots to the real system question.

---

### Question 3: "KV cache is 80% of GPU memory. How do you reduce it?"

**Weak answer:**
> "Use fp16 instead of fp32? Or compress the cache?"

**Problem:** It starts from precision before checking the architecture.

---

**Strong answer:**
> "I would first compute the raw KV payload:
> `layers × context × KV_heads × head_dim × 2(K,V) × bytes × active_requests`.
>
> Then I would look for the best lever in this order:
>
> 1. **Architecture**
>    - Is this MHA, GQA, or MQA?
>    - Moving from 64 KV heads to 8 KV heads is an exact 8x cache reduction at the same tensor shape otherwise.
>
> 2. **Workload**
>    - Are context lengths much longer than they need to be?
>    - Are we admitting too many active requests at once?
>
> 3. **Representation**
>    - KV quantization can reduce raw payload further.
>    - For the same tensor shape, going from 2 bytes/element to 1 byte/element halves the raw cache payload.
>
> 4. **Runtime policy**
>    - paging / reuse / eviction / admission control
>
> Example: in the 80-layer, 4K, batch-8 example from the previous section:
> - MHA with 64 KV heads: 80 GiB raw K+V
> - GQA with 8 KV heads: 10 GiB raw K+V
> - GQA at 1 byte per stored element: about 5 GiB raw payload before metadata
>
> So I would not jump straight to 'compress it'. I would first check whether the model architecture and concurrency policy are the larger lever."

**Why it's strong:** It attacks the problem in the right order and uses exact scaling laws.

---

### Question 4: "A batch of 32 prompts gets only 2x throughput vs a single prompt. Why not 32x?"

**Weak answer:**
> "Because there's overhead. Communication or something."

**Problem:** Too vague.

---

**Strong answer:**
> "Perfect 32x scaling would require the whole workload to be parallelizable and unconstrained by memory, padding, queueing, and decode.
>
> A 2x gain suggests that one of these is happening:
> - decode dominates end-to-end time, so batching prefill alone cannot deliver large gains
> - the batch is shaped by the longest prompt, so shorter requests waste work
> - memory bandwidth or KV pressure limits effective batch growth
> - the front-end scheduler is not actually keeping the device full
>
> I would separate:
> 1. pure prefill throughput
> 2. pure decode throughput
> 3. end-to-end request throughput
>
> Then I would profile where the scaling stopped."

**Why it's strong:** It explains why 'batch size' and 'system speedup' are not the same thing.

---

### Question 5: "Your inference cost is $10 per 1M tokens. A competitor says $2. What changes?"

**Weak answer:**
> "Better hardware? Optimized kernels?"

**Problem:** It assumes cost is mainly a kernel issue.

---

**Strong answer:**
> "A 5x cost gap usually means system design, model choice, or workload policy — not just one faster kernel.
>
> I would inspect:
> - achieved utilization
> - model size / routing strategy
> - quantization
> - prompt reuse opportunities
> - batching and admission policy
> - how much traffic really needs the large model
>
> A common pattern is:
> - smaller or quantized model for easy traffic
> - larger model only for hard traffic
> - better batching and reuse on top
>
> So my answer would be: first profile where cost comes from, then decide whether the biggest lever is utilization, architecture, routing, or model mix. I would not assume the answer is 'optimize attention'."

**Why it's strong:** It frames cost as a systems-and-product question, not just a kernel question.

---

### Meta-pattern: What strong infrastructure answers share

1. **Measure first** — do not optimize blindly
2. **Reason from scaling laws** — use layers, heads, context, bytes, and active requests
3. **Separate workload phases** — prefill, decode, queueing, and communication are different
4. **Use arithmetic carefully** — back-of-the-envelope math is powerful when the assumptions are explicit
5. **Name the real lever** — scheduler, KV footprint, model choice, routing, or kernel

If you can do those five things calmly in an interview, you will sound much stronger than someone who only remembers vocabulary.


### Exercise 2 — Why is inference slower across time than training?

**Hint:**  
Think about whether the future target tokens are known.



### Exercise 2 — Answer

During training, the whole sequence is available, so the model computes predictions for many time steps in one pass.

During inference, the next token is unknown until the previous token is generated.
So decoding becomes sequential across time:
- generate one token
- append it
- repeat

That reduces time-step parallelism.



### Exercise 3 — What exactly is stored in a KV cache?

**Hint:**  
Focus on the self-attention layers, not the whole model.



### Exercise 3 — Answer

For each decoder layer, the KV cache stores:
- past key tensors
- past value tensors

Typical shapes are:
- keys:   `[batch, heads, cached_len, head_dim]`
- values: `[batch, heads, cached_len, head_dim]`

The cache does **not** usually store every intermediate tensor in the whole model.
It stores the attention-side K/V needed to reuse old context efficiently.



### Exercise 4 — Why should KV cache usually be used only for inference?

**Hint:**  
Think about training behavior and sequence processing.



### Exercise 4 — Answer

Training usually processes full sequences in parallel under teacher-forced next-token prediction.

KV caching is mainly helpful for autoregressive inference, where generation happens step by step.

Using inference-style caching during training is usually unnecessary and can create complexity or incorrect assumptions about the training flow.



### Exercise 5 — Does KV cache make attention cost independent of context length?

**Hint:**  
Ask yourself whether the new query token still attends over the full past context.



### Exercise 5 — Answer

No.

KV cache avoids recomputing old K/V and old hidden-state paths, which saves a lot of work.

But the new query token still attends over all cached key positions, so the attention work for that token still grows with context length.



### Exercise 6 — Prefill vs decode

What is the difference between:
- **prefill**
- **decode**

**Hint:**  
One processes the prompt. The other extends the sequence one token at a time.



### Exercise 6 — Answer

**Prefill** is the initial prompt processing pass:
- run the whole prompt through the decoder
- build the initial per-layer KV cache

**Decode** is the iterative generation phase:
- process one new token at a time
- append new K/V into the cache
- generate the next token



### Exercise 7 — Small coding challenge

Modify the toy dataset so that instead of counting upward by `+1`, it counts upward by `+2` modulo 10.

Example:
- current pattern: `3 4 5 6`
- new pattern: `3 5 7 9`

**Hint:**  
Only the batch generator needs to change.  
Look for the line that computes each digit from `start_digit + i`.



### Exercise 7 — Answer

Inside the batch generator, change:

```python
digits = [DIGIT_OFFSET + ((start_digit + i) % 10) for i in range(seq_len)]
```

to:

```python
digits = [DIGIT_OFFSET + ((start_digit + 2 * i) % 10) for i in range(seq_len)]
```

The model architecture stays the same.
Only the data pattern changes.



### Exercise 8 — Another small coding challenge

Add a `temperature` argument to generation.

Right now generation uses greedy decoding:
- take the argmax token every step

How would you make it sample more creatively?

**Hint:**  
Divide logits by temperature and sample from a softmax distribution instead of taking argmax.



### Exercise 8 — Answer

A common pattern is:

```python
probs = torch.softmax(next_token_logits / temperature, dim=-1)
next_token = torch.multinomial(probs, num_samples=1)
```

Interpretation:
- lower temperature -> sharper distribution
- higher temperature -> flatter distribution

Greedy decoding is still useful for debugging because it is deterministic.



## 27. Advanced serving systems companion: the control plane around the model

Up to this point, the notebook has focused on the **model-side** mental model:
decoder-only execution, KV cache, paged KV blocks, and multi-GPU basics.

For L5/L6 inference-serving interviews, that is necessary but not sufficient.

You also need a strong mental model for the **serving control plane** around the model:

1. **scheduler token budgets and admission control**
2. **router / launcher / model-server architecture**
3. **queueing and request-level metrics**
4. **chunked prefill**
5. **preemption and recompute under KV pressure**
6. **host offload / cache reuse**
7. **speculative decoding**
8. **quantization tradeoffs**
9. **disaggregated prefill / decode**
10. **overload behavior and SLO protection**

### Important note on every number below

All timings, capacities, and throughput values in the rest of this section are:

- **toy teaching numbers**
- **production inspired**
- **interview oriented**
- **not benchmark claims**

That is deliberate.

The goal here is to build **intuition** and **decision-making skill**.
In a real job, you would validate these ideas with profiling, load tests, and hardware-specific measurements.



## 28. Router / launcher / model-server architecture: the hot path vs the orchestration path

A surprisingly common interview failure is mixing together:

- the **request hot path**
- the **process orchestration path**
- the **model execution path**

### Mental model 1 — TGI-style separation

A very clean architecture example is the Text Generation Inference mental model:

- **router / webserver** receives requests, queues them, batches them, and calls model servers
- **launcher** starts the router and one or more model-server processes
- **model server(s)** own the model shards and run the actual forward passes

Important interview point:

> the **launcher is not the main steady-state inference hot path**  
> it is mostly the **process orchestration / bring-up path**

### Mental model 2 — vLLM-style serving split

For vLLM V1, a useful mental model is:

- **API server**: request ingress, OpenAI-compatible serving, streaming back to clients
- **engine core**: scheduler, KV-cache management, work dispatch
- **GPU workers**: weight residency, forward passes, GPU memory ownership

Important interview point:

> if the interviewer asks about **where serving intelligence lives**,  
> the best place to start is usually the **engine core / scheduler side**, not the tokenizer and not the CUDA kernel alone

The diagram below is intentionally simplified, but it is close to the shape you should be able to explain on a whiteboard.


In [ ]:

# ============================================================
# Architecture diagrams for interview discussion
# ------------------------------------------------------------
# We draw two high-level mental models:
#   1) a TGI-style router / launcher / model-server split
#   2) a vLLM-style API server / engine core / GPU worker split
#
# These are not meant to be pixel-perfect copies of the official
# diagrams. They are "whiteboard quality" diagrams that you could
# reproduce in an interview discussion.
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

def draw_serving_architecture_diagrams():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # -----------------------------
    # Left panel: TGI-style design
    # -----------------------------
    ax = axes[0]
    ax.set_title("TGI-style mental model\n(router / launcher / model servers)")
    ax.axis("off")

    boxes = {
        "client":   (0.05, 0.45, 0.18, 0.12, "Client / SDK"),
        "router":   (0.30, 0.45, 0.20, 0.12, "Router / Webserver\nqueue + batching"),
        "model0":   (0.62, 0.60, 0.22, 0.12, "Model server shard 0"),
        "model1":   (0.62, 0.42, 0.22, 0.12, "Model server shard 1"),
        "launcher": (0.30, 0.75, 0.20, 0.12, "Launcher\nstarts processes"),
    }

    for _, (x, y, w, h, label) in boxes.items():
        patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02", facecolor="#eef3ff", edgecolor="black")
        ax.add_patch(patch)
        ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=10)

    arrows = [
        ("client", "router", False),
        ("router", "model0", False),
        ("router", "model1", False),
        ("launcher", "router", True),
        ("launcher", "model0", True),
        ("launcher", "model1", True),
    ]

    for src, dst, dashed in arrows:
        x1, y1, w1, h1, _ = boxes[src]
        x2, y2, w2, h2, _ = boxes[dst]

        if not dashed:
            start = (x1 + w1, y1 + h1 / 2)
            end = (x2, y2 + h2 / 2)
            style = "-"
        else:
            start = (x1 + w1 / 2, y1)
            end = (x2 + w2 / 2, y2 + h2)
            style = "--"

        arrow = FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=12, linewidth=1.3, linestyle=style)
        ax.add_patch(arrow)

    ax.text(0.05, 0.12, "Hot path:\nclient -> router -> model server(s)\n\nControl / bring-up path:\nlauncher starts router + model servers", fontsize=9)

    # -----------------------------
    # Right panel: vLLM-style design
    # -----------------------------
    ax = axes[1]
    ax.set_title("vLLM-style mental model\n(API server / engine core / GPU workers)")
    ax.axis("off")

    boxes = {
        "client": (0.05, 0.45, 0.18, 0.12, "Client / SDK"),
        "api":    (0.30, 0.45, 0.20, 0.12, "API server\nOpenAI-compatible"),
        "engine": (0.58, 0.45, 0.22, 0.12, "Engine core\nscheduler + KV"),
        "gpu0":   (0.58, 0.68, 0.18, 0.10, "GPU worker 0"),
        "gpu1":   (0.79, 0.68, 0.18, 0.10, "GPU worker 1"),
        "gpu2":   (0.58, 0.23, 0.18, 0.10, "GPU worker 2"),
        "gpu3":   (0.79, 0.23, 0.18, 0.10, "GPU worker 3"),
    }

    for _, (x, y, w, h, label) in boxes.items():
        patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02", facecolor="#f3f8ef", edgecolor="black")
        ax.add_patch(patch)
        ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=10)

    links = [
        ("client", "api"),
        ("api", "engine"),
        ("engine", "gpu0"),
        ("engine", "gpu1"),
        ("engine", "gpu2"),
        ("engine", "gpu3"),
    ]

    for src, dst in links:
        x1, y1, w1, h1, _ = boxes[src]
        x2, y2, w2, h2, _ = boxes[dst]

        if src == "engine":
            start = (x1 + w1 / 2, y1 + h1 if y2 > y1 else y1)
            end = (x2 + w2 / 2, y2 if y2 > y1 else y2 + h2)
        else:
            start = (x1 + w1, y1 + h1 / 2)
            end = (x2, y2 + h2 / 2)

        arrow = FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=12, linewidth=1.3)
        ax.add_patch(arrow)

    ax.text(0.05, 0.12, "Hot path:\nclient -> API server -> engine core -> GPU workers\n\nInterview cue:\nengine core is where you discuss scheduling,\nKV pressure, admission, and work dispatch", fontsize=9)

    plt.tight_layout()
    plt.show()
    plt.close(fig)

draw_serving_architecture_diagrams()



## 29. Scheduler token budgets and admission control

This is one of the most important serving topics for L5/L6 interviews.

### The big idea

A modern serving scheduler does **not** just ask:

> "Which request came first?"

It asks more operational questions:

- how many **tokens of work** can I safely issue this iteration?
- how many **active sequences** can I keep alive?
- do I protect short interactive requests from long RAG-style prompts?
- do I reject / defer / clamp requests when I am close to an SLO violation?
- do I let decodes run before big prefills?
- do I split a long prefill into **chunks**?

### Useful vLLM vocabulary to know

In official vLLM docs, the scheduler configuration includes ideas like:

- `max_num_batched_tokens`
- `max_num_scheduled_tokens`
- `max_num_seqs`
- `max_num_partial_prefills`
- `max_long_partial_prefills`
- `long_prefill_token_threshold`

You do **not** need to memorize every knob.
But you **do** need to understand the shape of the problem:

> the scheduler is turning **latency, memory, and fairness constraints** into a concrete per-iteration work plan

The next code cell defines a **toy serving scheduler** that we will reuse in several later sections.


In [ ]:

# ============================================================
# Shared helpers for the advanced serving sections
# ------------------------------------------------------------
# These helpers are intentionally verbose and heavily commented.
# The goal is not to mimic a production server perfectly.
# The goal is to create a compact teaching environment where:
#   - requests arrive over time
#   - prefills and decodes compete for budget
#   - queueing shows up in the metrics
#   - overload and admission control can be studied
# ============================================================

import math
import random
import copy
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from matplotlib.patches import Rectangle
import pandas as pd
from collections import deque

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)

@dataclass
class ToyServingRequest:
    """
    Small request object for teaching scheduler behavior.

    Important simplification:
    - 'prompt_tokens' is our prefill work
    - 'output_tokens' is our decode work
    - one scheduler step is one abstract scheduling iteration
    """
    name: str
    arrival_step: int
    prompt_tokens: int
    output_tokens: int
    traffic_class: str = "interactive"

    remaining_prompt: int = 0
    remaining_output: int = 0
    first_service_step: Optional[int] = None
    first_token_step: Optional[int] = None
    finish_step: Optional[int] = None
    decode_emission_steps: List[int] = field(default_factory=list)
    timeline: List[str] = field(default_factory=list)
    rejected: bool = False
    rejection_reason: Optional[str] = None
    total_prefill_work: int = 0
    total_decode_work: int = 0
    admission_step: Optional[int] = None

    def reset_runtime(self):
        self.remaining_prompt = self.prompt_tokens
        self.remaining_output = self.output_tokens
        self.first_service_step = None
        self.first_token_step = None
        self.finish_step = None
        self.decode_emission_steps = []
        self.timeline = []
        self.rejected = False
        self.rejection_reason = None
        self.total_prefill_work = 0
        self.total_decode_work = 0
        self.admission_step = None


def clone_requests(requests: List[ToyServingRequest]) -> List[ToyServingRequest]:
    cloned = copy.deepcopy(requests)
    for req in cloned:
        req.reset_runtime()
    return cloned


def nearest_rank(values: List[float], quantile: float) -> Optional[float]:
    if not values:
        return None
    ordered = sorted(values)
    index = max(0, math.ceil(quantile * len(ordered)) - 1)
    return ordered[index]


def rows_to_df(rows: List[Dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    rename_map = {
        "class_": "class",
        "queue": "queue_steps",
        "ttft": "ttft_steps",
        "avg_itl": "avg_itl_steps",
        "e2e": "end_to_end_steps",
    }
    return df.rename(columns=rename_map)


STATE_COLORS = {
    ".": "#ffffff",
    "Q": "#cfcfcf",
    "P": "#4c78a8",
    "p": "#9ecae9",
    "D": "#59a14f",
    "d": "#a1d99b",
    "X": "#f28e2b",
    "T": "#b07aa1",
    "R": "#e15759",
}


def plot_timelines(requests: List[ToyServingRequest], title: str):
    if not requests:
        return

    max_steps = max(len(req.timeline) for req in requests)
    fig, ax = plt.subplots(figsize=(12, 0.6 * len(requests) + 1.5))

    for row_idx, req in enumerate(requests):
        y = len(requests) - 1 - row_idx
        for step, state in enumerate(req.timeline):
            rect = Rectangle((step, y), 1, 0.8, facecolor=STATE_COLORS.get(state, "#dddddd"), edgecolor="black", linewidth=0.3)
            ax.add_patch(rect)
        ax.text(-0.2, y + 0.4, req.name, ha="right", va="center", fontsize=10)

    ax.set_xlim(0, max_steps)
    ax.set_ylim(0, len(requests))
    ax.set_xticks(range(max_steps + 1))
    ax.set_yticks([])
    ax.set_title(title)

    legend_keys = ["Q", "P", "D", "X", "T", "R"]
    for i, key in enumerate(legend_keys):
        y = len(requests) - 1 - i * 0.35
        ax.add_patch(Rectangle((max_steps + 0.5, y), 0.25, 0.2, facecolor=STATE_COLORS[key], edgecolor="black"))
        ax.text(max_steps + 0.8, y + 0.1, key, va="center", fontsize=8)

    ax.text(max_steps + 1.3, len(requests) - 0.9, "Q=queue, P=prefill, D=decode, X=preempted, T=transfer, R=rejected", fontsize=8, va="top")
    ax.axis("off")
    plt.show()
    plt.close(fig)


def summarize_requests(requests: List[ToyServingRequest]) -> List[Dict]:
    rows = []
    for req in requests:
        if len(req.decode_emission_steps) >= 2:
            gaps = [later - earlier for earlier, later in zip(req.decode_emission_steps, req.decode_emission_steps[1:])]
            avg_itl = sum(gaps) / len(gaps)
        else:
            avg_itl = None

        rows.append({
            "name": req.name,
            "class_": req.traffic_class,
            "arrival": req.arrival_step,
            "prompt": req.prompt_tokens,
            "output": req.output_tokens,
            "rejected": req.rejected,
            "queue": None if req.first_service_step is None else req.first_service_step - req.arrival_step,
            "ttft": None if req.first_token_step is None else req.first_token_step - req.arrival_step,
            "avg_itl": avg_itl,
            "e2e": None if req.finish_step is None else req.finish_step - req.arrival_step + 1,
            "prefill_work": req.total_prefill_work,
            "decode_work": req.total_decode_work,
            "timeline": "".join(req.timeline),
        })
    return rows


def print_metric_summary(requests: List[ToyServingRequest], label: str):
    rows = summarize_requests(requests)
    df = rows_to_df(rows)
    admitted = df[(df["rejected"] == False) & (df["end_to_end_steps"].notna())]
    ttfts = [float(x) for x in admitted["ttft_steps"].dropna().tolist()]
    e2es = [float(x) for x in admitted["end_to_end_steps"].dropna().tolist()]

    print(f"\n=== {label} ===")
    print(df[["name", "class", "arrival", "prompt", "output", "rejected", "queue_steps", "ttft_steps", "avg_itl_steps", "end_to_end_steps"]].to_string(index=False))

    if ttfts:
        print(f"\nTTFT summary (scheduler-step units): p50={nearest_rank(ttfts, 0.50)}, p95={nearest_rank(ttfts, 0.95)}")
    if e2es:
        print(f"End-to-end summary (scheduler-step units): p50={nearest_rank(e2es, 0.50)}, p95={nearest_rank(e2es, 0.95)}")


def simulate_monolithic_scheduler(
    requests: List[ToyServingRequest],
    max_batch_tokens: int = 12,
    max_active_seqs: int = 4,
    prefill_mode: str = "blocking",
    prefill_chunk_size: int = 4,
    queue_cap: Optional[int] = None,
    max_steps: int = 200,
    sort_prefill_short_first: bool = False,
):
    reqs = clone_requests(requests)
    by_arrival: Dict[int, List[ToyServingRequest]] = {}
    for req in reqs:
        by_arrival.setdefault(req.arrival_step, []).append(req)

    waiting: List[ToyServingRequest] = []
    engine: List[ToyServingRequest] = []
    done_count = 0
    step = 0
    rr_decode_index = 0
    event_log: List[str] = []

    while step < max_steps and done_count < len(reqs):
        # Default timeline marker for every request at this step.
        for req in reqs:
            marker = "."
            if req.rejected:
                marker = "R"
            elif req.finish_step is not None:
                marker = "."
            elif step < req.arrival_step:
                marker = "."
            elif req in waiting:
                marker = "Q"
            elif req in engine:
                marker = "p" if req.remaining_prompt > 0 else "d"
            req.timeline.append(marker)

        # Inject newly arrived requests.
        for req in by_arrival.get(step, []):
            if queue_cap is not None and len(waiting) >= queue_cap:
                req.rejected = True
                req.rejection_reason = "queue_cap"
                req.timeline[-1] = "R"
                done_count += 1
                event_log.append(f"step {step}: reject {req.name} because the queue is full")
            else:
                waiting.append(req)

        # Admit queued requests into the engine if there is room.
        while waiting and len(engine) < max_active_seqs:
            req = waiting.pop(0)
            req.admission_step = step
            engine.append(req)

        budget = max_batch_tokens

        def mark(req: ToyServingRequest, char: str):
            req.timeline[-1] = char

        if prefill_mode == "blocking":
            prefill_candidates = [req for req in engine if req.remaining_prompt > 0]
            if sort_prefill_short_first:
                prefill_candidates.sort(key=lambda req: (req.remaining_prompt, req.arrival_step, req.name))

            if prefill_candidates:
                req = prefill_candidates[0]
                if req.first_service_step is None:
                    req.first_service_step = step
                work = min(req.remaining_prompt, budget)
                req.remaining_prompt -= work
                req.total_prefill_work += work
                mark(req, "P")
                if req.remaining_prompt == 0:
                    event_log.append(f"step {step}: {req.name} finished prefill")
            else:
                decoders = [req for req in engine if req.remaining_prompt == 0 and req.remaining_output > 0]
                if decoders:
                    rr_decode_index %= len(decoders)
                    start_index = rr_decode_index
                    while budget > 0 and decoders:
                        req = decoders[rr_decode_index % len(decoders)]
                        if req.first_service_step is None:
                            req.first_service_step = step
                        req.remaining_output -= 1
                        req.total_decode_work += 1
                        req.decode_emission_steps.append(step)
                        if req.first_token_step is None:
                            req.first_token_step = step
                        mark(req, "D")
                        budget -= 1
                        rr_decode_index = (rr_decode_index + 1) % len(decoders)
                        if req.remaining_output == 0:
                            req.finish_step = step
                            engine.remove(req)
                            done_count += 1
                            event_log.append(f"step {step}: {req.name} finished decode")
                            decoders = [candidate for candidate in engine if candidate.remaining_prompt == 0 and candidate.remaining_output > 0]
                            if decoders:
                                rr_decode_index %= len(decoders)
                        if rr_decode_index == start_index:
                            break
        elif prefill_mode == "chunked_decode_priority":
            decoders = [req for req in engine if req.remaining_prompt == 0 and req.remaining_output > 0]
            if decoders:
                rr_decode_index %= len(decoders)
                start_index = rr_decode_index
                for offset in range(len(decoders)):
                    if budget <= 0:
                        break
                    req = decoders[(start_index + offset) % len(decoders)]
                    if req.first_service_step is None:
                        req.first_service_step = step
                    req.remaining_output -= 1
                    req.total_decode_work += 1
                    req.decode_emission_steps.append(step)
                    if req.first_token_step is None:
                        req.first_token_step = step
                    mark(req, "D")
                    budget -= 1
                    if req.remaining_output == 0:
                        req.finish_step = step
                        engine.remove(req)
                        done_count += 1
                        event_log.append(f"step {step}: {req.name} finished decode")
                rr_decode_index += 1

            prefills = [req for req in engine if req.remaining_prompt > 0]
            if sort_prefill_short_first:
                prefills.sort(key=lambda req: (req.remaining_prompt, req.arrival_step, req.name))
            for req in prefills:
                if budget <= 0:
                    break
                if req.first_service_step is None:
                    req.first_service_step = step
                work = min(prefill_chunk_size, req.remaining_prompt, budget)
                req.remaining_prompt -= work
                req.total_prefill_work += work
                mark(req, "P")
                budget -= work
                if req.remaining_prompt == 0:
                    event_log.append(f"step {step}: {req.name} finished prefill")
        else:
            raise ValueError(f"Unknown prefill_mode: {prefill_mode}")

        step += 1

    return reqs, event_log


In [ ]:

# ============================================================
# First workload: mixed traffic under a budgeted scheduler
# ============================================================

mixed_requests = [
    ToyServingRequest("chat_0", arrival_step=0, prompt_tokens=4,  output_tokens=12, traffic_class="chat"),
    ToyServingRequest("rag_1",  arrival_step=1, prompt_tokens=14, output_tokens=6,  traffic_class="rag"),
    ToyServingRequest("chat_2", arrival_step=2, prompt_tokens=4,  output_tokens=10, traffic_class="chat"),
    ToyServingRequest("tool_3", arrival_step=3, prompt_tokens=6,  output_tokens=5,  traffic_class="tool"),
    ToyServingRequest("chat_4", arrival_step=4, prompt_tokens=5,  output_tokens=8,  traffic_class="chat"),
    ToyServingRequest("rag_5",  arrival_step=5, prompt_tokens=16, output_tokens=7,  traffic_class="rag"),
]

scheduled_requests, schedule_log = simulate_monolithic_scheduler(
    mixed_requests,
    max_batch_tokens=12,
    max_active_seqs=4,
    prefill_mode="chunked_decode_priority",
    prefill_chunk_size=4,
)

print_metric_summary(scheduled_requests, label="Mixed workload under a token-budgeted scheduler")
plot_timelines(scheduled_requests, title="Mixed traffic timeline: queueing + chunked prefill + decode")



## 30. Queueing and request-level metrics

A lot of candidates know model internals but cannot talk cleanly about **request-level SLO metrics**.

That is a red flag for infra interviews.

### Metrics you should be comfortable defining

- **queue time**: how long the request waits before it receives service
- **TTFT** (time to first token): how long until the first output token is emitted
- **ITL** (inter-token latency): gap between output tokens during decode
- **end-to-end latency**: arrival to last token
- **throughput**: requests / second or tokens / second
- **p50 / p95 / p99**: distribution summaries, not just averages

### Important interview point

A system can have:

- decent average throughput
- acceptable p50 latency
- terrible p95 TTFT

...and still feel bad to users.

In serving interviews, you should talk about **distribution shape**, not just means.


In [ ]:

# ============================================================
# Metric view for the mixed workload
# ============================================================

mixed_df = rows_to_df(summarize_requests(scheduled_requests))

print("Per-request metrics:")
print(mixed_df[["name", "class", "prompt", "output", "queue_steps", "ttft_steps", "avg_itl_steps", "end_to_end_steps"]].to_string(index=False))

ttft_values = [float(x) for x in mixed_df["ttft_steps"].dropna().tolist()]
e2e_values = [float(x) for x in mixed_df["end_to_end_steps"].dropna().tolist()]

summary_df = pd.DataFrame([
    {"metric": "TTFT", "p50": nearest_rank(ttft_values, 0.50), "p95": nearest_rank(ttft_values, 0.95), "mean": round(sum(ttft_values) / len(ttft_values), 2)},
    {"metric": "end_to_end", "p50": nearest_rank(e2e_values, 0.50), "p95": nearest_rank(e2e_values, 0.95), "mean": round(sum(e2e_values) / len(e2e_values), 2)},
])

print("\nPercentile summary:")
print(summary_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(summary_df))
ax.bar([i - 0.2 for i in x], summary_df["p50"], width=0.2, label="p50")
ax.bar([i for i in x], summary_df["p95"], width=0.2, label="p95")
ax.bar([i + 0.2 for i in x], summary_df["mean"], width=0.2, label="mean")
ax.set_xticks(list(x))
ax.set_xticklabels(summary_df["metric"].tolist())
ax.set_ylabel("scheduler-step units")
ax.set_title("Request-level metric summary (toy workload)")
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)



## 31. Chunked prefill: why it exists

This is a very important interview topic.

### The core problem

A long prompt is expensive to prefill.

If you let a long prefill monopolize the scheduler, it can:

- delay short interactive requests
- stretch decode gaps for requests already streaming
- create a bad p95 user experience even when average throughput looks okay

### The chunked-prefill idea

Instead of treating a long prompt as one indivisible wall of work, the scheduler can split it into **smaller prefill chunks**.

That creates room to:

- keep already-active decodes moving
- let short prefills jump in
- use leftover budget more flexibly

### Important tradeoff

Chunked prefill is not "free magic".

Depending on the policy, it can:

- improve **ITL / responsiveness for active streams**
- sometimes worsen **TTFT for the long prompt itself**
- change fairness between short and long requests

That tradeoff is exactly the kind of thing a senior candidate should be able to explain.


In [ ]:

# ============================================================
# Blocking-prefill vs chunked-prefill comparison
# ============================================================

chunked_demo_requests = [
    ToyServingRequest("chat_A",  arrival_step=0, prompt_tokens=8,  output_tokens=10, traffic_class="chat"),
    ToyServingRequest("rag_B",   arrival_step=3, prompt_tokens=10, output_tokens=5,  traffic_class="rag"),
    ToyServingRequest("short_C", arrival_step=4, prompt_tokens=3,  output_tokens=4,  traffic_class="chat"),
]

blocking_results, _ = simulate_monolithic_scheduler(
    chunked_demo_requests,
    max_batch_tokens=12,
    max_active_seqs=3,
    prefill_mode="blocking",
    prefill_chunk_size=4,
)

chunked_results, _ = simulate_monolithic_scheduler(
    chunked_demo_requests,
    max_batch_tokens=12,
    max_active_seqs=3,
    prefill_mode="chunked_decode_priority",
    prefill_chunk_size=4,
)

print_metric_summary(blocking_results, label="Blocking-prefill policy")
plot_timelines(blocking_results, title="Blocking-prefill policy")

print_metric_summary(chunked_results, label="Chunked-prefill + decode-priority policy")
plot_timelines(chunked_results, title="Chunked-prefill + decode-priority policy")

blocking_df = rows_to_df(summarize_requests(blocking_results))
chunked_df = rows_to_df(summarize_requests(chunked_results))
comparison = blocking_df[["name", "ttft_steps", "avg_itl_steps", "end_to_end_steps"]].merge(
    chunked_df[["name", "ttft_steps", "avg_itl_steps", "end_to_end_steps"]],
    on="name",
    suffixes=("_blocking", "_chunked"),
)

print("\nSide-by-side comparison:")
print(comparison.to_string(index=False))



## 32. Preemption and recompute under KV pressure

Another high-value interview topic.

### What preemption usually means

If the serving system warns that requests are being **preempted** because KV space is tight, that usually means:

- concurrency is high
- context lengths are high
- active sequences are competing for limited KV capacity
- the system cannot keep every live sequence resident in GPU memory at once

### Why recompute appears

If a request loses its live KV state, the system may later need to **rebuild** that context before it can continue decoding.

That means:

- extra prefill work
- worse tail latency
- throughput wasted on rebuilding old state instead of serving new work

### Important interview point

When you hear "preemption warnings", do **not** jump straight to:

> "We need a faster attention kernel."

The first question should usually be:

> "Is this really a **KV-capacity** problem?"

The toy simulation below intentionally triggers that behavior.


In [ ]:

# ============================================================
# Toy KV-pressure simulator
# ============================================================

@dataclass
class KVPressureRequest:
    name: str
    arrival_step: int
    prompt_tokens: int
    output_tokens: int

    remaining_prefill: int = 0
    remaining_output: int = 0
    built_context_tokens: int = 0
    total_prefill_work: int = 0
    total_decode_work: int = 0
    total_kv_rebuilt: int = 0
    preemptions: int = 0
    first_token_step: Optional[int] = None
    finish_step: Optional[int] = None
    timeline: List[str] = field(default_factory=list)

    def reset_runtime(self):
        self.remaining_prefill = self.prompt_tokens
        self.remaining_output = self.output_tokens
        self.built_context_tokens = 0
        self.total_prefill_work = 0
        self.total_decode_work = 0
        self.total_kv_rebuilt = 0
        self.preemptions = 0
        self.first_token_step = None
        self.finish_step = None
        self.timeline = []


def simulate_kv_pressure(
    requests: List[KVPressureRequest],
    prefill_chunk: int = 4,
    decode_budget: int = 2,
    kv_capacity: int = 30,
    max_active: int = 3,
    max_steps: int = 100,
):
    reqs = copy.deepcopy(requests)
    for req in reqs:
        req.reset_runtime()

    by_arrival: Dict[int, List[KVPressureRequest]] = {}
    for req in reqs:
        by_arrival.setdefault(req.arrival_step, []).append(req)

    waiting = deque()
    active: List[KVPressureRequest] = []
    paused: List[KVPressureRequest] = []
    done_count = 0
    step = 0
    event_log: List[str] = []

    def live_kv() -> int:
        return sum(req.built_context_tokens for req in active)

    while step < max_steps and done_count < len(reqs):
        for req in reqs:
            marker = "."
            if req.finish_step is not None:
                marker = "."
            elif step < req.arrival_step:
                marker = "."
            elif req in waiting:
                marker = "Q"
            elif req in paused:
                marker = "X"
            elif req in active:
                marker = "d" if req.remaining_prefill == 0 else "p"
            req.timeline.append(marker)

        for req in by_arrival.get(step, []):
            waiting.append(req)

        while waiting and len(active) < max_active:
            active.append(waiting.popleft())
        while paused and len(active) < max_active:
            active.append(paused.pop(0))

        decode_tokens_left = decode_budget
        for req in list(active):
            if decode_tokens_left <= 0:
                break
            if req.remaining_prefill > 0 or req.remaining_output == 0:
                continue

            if live_kv() + 1 > kv_capacity:
                victim = max(active, key=lambda candidate: candidate.built_context_tokens)
                active.remove(victim)
                victim.preemptions += 1
                victim.total_kv_rebuilt += victim.built_context_tokens
                victim.remaining_prefill += victim.built_context_tokens
                victim.built_context_tokens = 0
                paused.append(victim)
                victim.timeline[-1] = "X"
                event_log.append(f"step {step}: preempt {victim.name}; later it must rebuild {victim.total_kv_rebuilt} tokens of KV context")
                continue

            req.remaining_output -= 1
            req.total_decode_work += 1
            req.built_context_tokens += 1
            req.timeline[-1] = "D"
            if req.first_token_step is None:
                req.first_token_step = step
            decode_tokens_left -= 1
            if req.remaining_output == 0:
                req.finish_step = step
                active.remove(req)
                done_count += 1
                event_log.append(f"step {step}: {req.name} finished")

        for req in list(active):
            if req.remaining_prefill <= 0:
                continue
            local_work = 0
            while local_work < prefill_chunk and req.remaining_prefill > 0:
                if live_kv() + 1 > kv_capacity:
                    victim = max(active, key=lambda candidate: candidate.built_context_tokens)
                    active.remove(victim)
                    victim.preemptions += 1
                    victim.total_kv_rebuilt += victim.built_context_tokens
                    victim.remaining_prefill += victim.built_context_tokens
                    victim.built_context_tokens = 0
                    paused.append(victim)
                    victim.timeline[-1] = "X"
                    event_log.append(f"step {step}: preempt {victim.name}; later it must rebuild {victim.total_kv_rebuilt} tokens of KV context")
                    if victim is req:
                        break
                    else:
                        continue

                req.remaining_prefill -= 1
                req.total_prefill_work += 1
                req.built_context_tokens += 1
                req.timeline[-1] = "P"
                local_work += 1

        step += 1

    return reqs, event_log


kv_pressure_requests = [
    KVPressureRequest("A_long",  arrival_step=0, prompt_tokens=12, output_tokens=6),
    KVPressureRequest("B_mid",   arrival_step=1, prompt_tokens=8,  output_tokens=6),
    KVPressureRequest("C_short", arrival_step=2, prompt_tokens=4,  output_tokens=4),
]

kv_pressure_results, kv_pressure_log = simulate_kv_pressure(
    kv_pressure_requests,
    prefill_chunk=4,
    decode_budget=2,
    kv_capacity=30,
)

summary_rows = []
for req in kv_pressure_results:
    summary_rows.append({
        "name": req.name,
        "preemptions": req.preemptions,
        "kv_tokens_rebuilt": req.total_kv_rebuilt,
        "first_token_step": req.first_token_step,
        "finish_step": req.finish_step,
        "prefill_work": req.total_prefill_work,
        "decode_work": req.total_decode_work,
        "timeline": "".join(req.timeline),
    })

kv_pressure_df = pd.DataFrame(summary_rows)
print("KV-pressure summary:")
print(kv_pressure_df.to_string(index=False))

print("\nEvent log:")
for line in kv_pressure_log:
    print(" -", line)

plot_timelines(kv_pressure_results, title="Preemption and recompute under KV pressure")



## 33. Host offload and cache reuse

This topic is related to PagedAttention, but it is not the same thing.

### Two separate ideas

#### 1. Prefix / KV reuse
If many requests share the same prefix, a serving system may be able to reuse previously built KV state instead of recomputing the whole prefix from scratch.

That is great for:

- repeated system prompts
- repeated multi-turn chat prefixes
- repeated template prompts
- certain agent / workflow patterns

#### 2. Host offload
If GPU memory is tight, some KV state can be moved to **CPU memory** or another storage tier.

That can improve effective reuse / capacity, but it introduces a different tradeoff:

- GPU hit: fastest
- CPU restore: slower than a GPU hit, but often much better than a full cold prefill
- cold miss: most expensive

### Important interview point

If a system says "we support KV offload", the right reaction is **not**:

> "Great, memory problem solved."

The right reaction is:

> "Good, now tell me the restore path cost, hit rate, and whether TTFT improves enough to justify the extra complexity."

The toy cell below uses simple units to compare those three cases.


In [ ]:

# ============================================================
# Toy model for cold miss vs GPU reuse vs CPU restore
# ============================================================

def prefix_reuse_scenarios(
    shared_prefix_tokens: int = 800,
    unique_tokens: int = 120,
    cold_prefill_cost_per_token: float = 1.00,
    gpu_reuse_cost_per_token: float = 0.05,
    cpu_restore_cost_per_token: float = 0.25,
):
    cold_miss = (shared_prefix_tokens + unique_tokens) * cold_prefill_cost_per_token
    gpu_hit = unique_tokens * cold_prefill_cost_per_token + shared_prefix_tokens * gpu_reuse_cost_per_token
    cpu_restore = unique_tokens * cold_prefill_cost_per_token + shared_prefix_tokens * cpu_restore_cost_per_token
    return pd.DataFrame([
        {"scenario": "cold miss",            "ttft_units": cold_miss},
        {"scenario": "GPU prefix reuse hit", "ttft_units": gpu_hit},
        {"scenario": "CPU offload restore",  "ttft_units": cpu_restore},
    ])

reuse_df = prefix_reuse_scenarios()
print("Toy TTFT comparison:")
print(reuse_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(reuse_df["scenario"], reuse_df["ttft_units"])
ax.set_ylabel("toy TTFT work units")
ax.set_title("Cold miss vs GPU reuse vs CPU restore")
plt.xticks(rotation=12, ha="right")
plt.tight_layout()
plt.show()
plt.close(fig)



## 34. Speculative decoding

Another strong L5/L6 topic.

### The intuition

The expensive model is the **verifier / target model**.

A cheaper draft mechanism proposes several future tokens.
If many of those proposals are accepted, the target model can advance the sequence faster than the usual "one expensive step per token" pattern.

### The catch

Speculative decoding helps only when the regime is right.

If:

- acceptance is low
- draft cost is too high
- the workload is already in the wrong saturation regime

...you may pay extra work without getting much latency benefit.

### Important interview point

A strong answer usually includes the phrase:

> "I would check the **acceptance rate** and the **extra draft overhead** before assuming speculation is helping."

The toy simulation below makes that tradeoff visible.


In [ ]:

# ============================================================
# Toy speculative decoding simulator
# ============================================================

def simulate_speculative_decoding(
    total_output_tokens: int = 128,
    draft_k: int = 4,
    accept_prob: float = 0.7,
    draft_cost_per_token: float = 0.15,
    target_cost_per_pass: float = 1.0,
    seed: int = 7,
):
    random.seed(seed)
    emitted = 0
    target_passes = 0
    draft_tokens = 0
    accepted_tokens = 0
    rejected_tokens = 0

    while emitted < total_output_tokens:
        proposal_len = min(draft_k, total_output_tokens - emitted)
        draft_tokens += proposal_len

        accepted_this_round = 0
        for _ in range(proposal_len):
            if random.random() < accept_prob:
                accepted_this_round += 1
            else:
                break

        target_passes += 1

        if accepted_this_round == proposal_len:
            emitted += proposal_len
            accepted_tokens += proposal_len
        else:
            emitted += accepted_this_round + 1
            accepted_tokens += accepted_this_round
            rejected_tokens += 1

    baseline_cost = total_output_tokens * target_cost_per_pass
    speculative_cost = draft_tokens * draft_cost_per_token + target_passes * target_cost_per_pass

    return {
        "accept_prob": accept_prob,
        "draft_k": draft_k,
        "output_tokens": total_output_tokens,
        "target_passes": target_passes,
        "draft_tokens": draft_tokens,
        "accepted_tokens": accepted_tokens,
        "rejected_tokens": rejected_tokens,
        "accepted_per_target_pass": accepted_tokens / target_passes if target_passes else None,
        "baseline_cost": baseline_cost,
        "speculative_cost": speculative_cost,
        "speedup_vs_baseline": baseline_cost / speculative_cost,
    }

spec_rows = []
for accept_prob in [0.20, 0.50, 0.80, 0.95]:
    spec_rows.append(simulate_speculative_decoding(total_output_tokens=128, draft_k=4, accept_prob=accept_prob, draft_cost_per_token=0.15, target_cost_per_pass=1.0, seed=7))

spec_df = pd.DataFrame(spec_rows)
print("Speculative-decoding tradeoff table:")
print(spec_df[["accept_prob", "target_passes", "draft_tokens", "accepted_tokens", "rejected_tokens", "accepted_per_target_pass", "speedup_vs_baseline"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(spec_df["accept_prob"], spec_df["speedup_vs_baseline"], marker="o")
ax.set_xlabel("acceptance probability")
ax.set_ylabel("toy speedup vs baseline")
ax.set_title("When speculation helps: toy view")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)



## 35. Quantization tradeoffs

Another topic that often separates strong systems candidates from slogan-driven answers.

### Good mental model

Quantization changes at least three things:

1. **weight footprint**
2. **kernel path / arithmetic path**
3. **system balance**

That means a smaller model payload does **not** automatically guarantee lower end-to-end latency.

### Why latency can fail to improve

Even if a model now "fits" after quantization, you may still be limited by:

- dequant overhead
- communication overhead
- KV / memory pressure
- weaker optimized-kernel support on that hardware
- scheduler / queueing effects that dominate the gain

### Important interview point

A strong answer sounds like this:

> "I would separate the **memory win** from the **latency win** and verify the actual kernel path being used."

The code below keeps the **memory math exact** for raw weight payload, but keeps the latency side explicitly labeled as a **toy systems sketch**.


In [ ]:

# ============================================================
# Quantization tradeoffs: exact raw-weight math + toy latency sketch
# ============================================================

def raw_weight_payload_gib(params_billions: float, bits_per_weight: int) -> float:
    params = params_billions * 1_000_000_000
    total_bytes = params * bits_per_weight / 8
    return total_bytes / (1024 ** 3)

precision_rows = [
    ("fp16", 16, 1.00, 0.00, 0.30),
    ("fp8",   8, 0.55, 0.05, 0.30),
    ("int8",  8, 0.60, 0.10, 0.30),
    ("int4",  4, 0.38, 0.32, 0.30),
]

quant_rows = []
for precision, bits, weight_term, dequant_term, comm_term in precision_rows:
    quant_rows.append({
        "precision": precision,
        "bits_per_weight": bits,
        "raw_weight_payload_gib": round(raw_weight_payload_gib(70, bits), 2),
        "toy_weight_term": weight_term,
        "toy_dequant_term": dequant_term,
        "toy_comm_or_kv_term": comm_term,
        "toy_total_latency_units": round(weight_term + dequant_term + comm_term, 2),
    })

quant_df = pd.DataFrame(quant_rows)
print("Raw memory math + toy latency sketch:")
print(quant_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(quant_df["precision"], quant_df["raw_weight_payload_gib"])
axes[0].set_ylabel("GiB")
axes[0].set_title("Exact raw weight payload for a 70B model")

x = range(len(quant_df))
axes[1].bar(x, quant_df["toy_weight_term"], label="weight-read term")
axes[1].bar(x, quant_df["toy_dequant_term"], bottom=quant_df["toy_weight_term"], label="dequant / kernel-path term")
axes[1].bar(x, quant_df["toy_comm_or_kv_term"], bottom=quant_df["toy_weight_term"] + quant_df["toy_dequant_term"], label="comm / KV / other term")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(quant_df["precision"].tolist())
axes[1].set_ylabel("toy latency units")
axes[1].set_title("Toy reminder: smaller weights != automatic lower latency")
axes[1].legend()
plt.tight_layout()
plt.show()
plt.close(fig)



## 36. Disaggregated prefill / decode

This is a concept worth knowing even if you do not implement it yourself.

### The main idea

Instead of forcing one serving pool to handle both phases, you can split the world into:

- **prefill-heavy capacity**
- **decode-heavy capacity**

That gives you more freedom to tune:

- **TTFT** on the prefill side
- **ITL** on the decode side

### Why people do this

Long prompts and long generations stress the system differently.

A single shared pool often forces a tradeoff like:

- give more budget to prefill -> better TTFT, worse decode smoothness
- give more budget to decode -> better ITL, worse prompt-side responsiveness

Disaggregated prefill / decode is attractive because it separates those knobs.

### Important interview point

A strong answer also mentions the cost:

- moving KV state between stages
- operational complexity
- extra failure modes
- possible throughput tradeoffs depending on the workload

The code below uses **toy napkin math** to show the tuning intuition, not to claim benchmark numbers.


In [ ]:

# ============================================================
# Toy napkin math for monolithic vs disaggregated tuning
# ============================================================

def toy_monolithic_latency(prompt_tokens: int, output_tokens: int, prefill_tokens_per_step: int, decode_tokens_per_step: int, active_decoders: int):
    ttft_steps = math.ceil(prompt_tokens / prefill_tokens_per_step)
    avg_itl_steps = active_decoders / decode_tokens_per_step
    e2e_steps = ttft_steps + math.ceil(output_tokens * avg_itl_steps)
    return ttft_steps, avg_itl_steps, e2e_steps


def toy_disaggregated_latency(prompt_tokens: int, output_tokens: int, prefill_tokens_per_step: int, decode_tokens_per_step: int, active_decoders: int, transfer_penalty_steps: int):
    ttft_steps = math.ceil(prompt_tokens / prefill_tokens_per_step) + transfer_penalty_steps
    avg_itl_steps = active_decoders / decode_tokens_per_step
    e2e_steps = ttft_steps + math.ceil(output_tokens * avg_itl_steps)
    return ttft_steps, avg_itl_steps, e2e_steps

prompt_tokens = 30
output_tokens = 18
active_decoders = 4
comparison_rows = []

ttft, itl, e2e = toy_monolithic_latency(prompt_tokens, output_tokens, prefill_tokens_per_step=4, decode_tokens_per_step=6, active_decoders=active_decoders)
comparison_rows.append({"policy": "monolithic_itl_favoring", "ttft_steps": ttft, "avg_itl_steps": round(itl, 2), "end_to_end_steps": e2e})

ttft, itl, e2e = toy_monolithic_latency(prompt_tokens, output_tokens, prefill_tokens_per_step=10, decode_tokens_per_step=2, active_decoders=active_decoders)
comparison_rows.append({"policy": "monolithic_ttft_favoring", "ttft_steps": ttft, "avg_itl_steps": round(itl, 2), "end_to_end_steps": e2e})

ttft, itl, e2e = toy_disaggregated_latency(prompt_tokens, output_tokens, prefill_tokens_per_step=10, decode_tokens_per_step=6, active_decoders=active_decoders, transfer_penalty_steps=1)
comparison_rows.append({"policy": "disaggregated_prefill_decode", "ttft_steps": ttft, "avg_itl_steps": round(itl, 2), "end_to_end_steps": e2e})

disagg_df = pd.DataFrame(comparison_rows)
print("Toy comparison:")
print(disagg_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(disagg_df["policy"], disagg_df["ttft_steps"])
axes[0].set_ylabel("TTFT steps")
axes[0].set_title("Prompt-side responsiveness")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(disagg_df["policy"], disagg_df["avg_itl_steps"])
axes[1].set_ylabel("average ITL steps")
axes[1].set_title("Decode-side smoothness")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()
plt.close(fig)



## 37. Overload behavior and SLO protection

This is where systems judgment matters.

### The naive policy

> accept everything, let it queue, and hope the system drains

That sounds customer-friendly, but under real overload it can be the worst option.

Why?

Because if the queue grows without protection, you may destroy:

- p95 TTFT
- admitted-user experience
- interactive traffic quality
- downstream SLOs

### Mature overload thinking

A stronger serving answer usually includes some combination of:

- queue caps
- traffic shaping
- max-output tightening
- separate async lane for long jobs
- admission control based on SLO risk
- early reject / retry / degrade strategies

### Important interview point

A strong candidate can explain why:

> protecting the latency of **admitted** traffic can be better than silently degrading everyone

The toy comparison below contrasts **blind queueing** vs **protective admission control**.


In [ ]:

# ============================================================
# Overload comparison: infinite queue vs queue cap
# ============================================================

overload_requests = [
    ToyServingRequest(
        name=f"req_{i}",
        arrival_step=i // 2,
        prompt_tokens=6 + (i % 3),
        output_tokens=8 + (i % 4),
        traffic_class="interactive",
    )
    for i in range(12)
]

blind_results, _ = simulate_monolithic_scheduler(
    overload_requests,
    max_batch_tokens=10,
    max_active_seqs=3,
    prefill_mode="chunked_decode_priority",
    prefill_chunk_size=4,
    queue_cap=None,
)

protected_results, _ = simulate_monolithic_scheduler(
    overload_requests,
    max_batch_tokens=10,
    max_active_seqs=3,
    prefill_mode="chunked_decode_priority",
    prefill_chunk_size=4,
    queue_cap=4,
)

print_metric_summary(blind_results, label="Blind queueing (accept everything)")
plot_timelines(blind_results, title="Blind queueing under overload")

print_metric_summary(protected_results, label="Protective admission (queue cap)")
plot_timelines(protected_results, title="Protective admission under overload")

def admitted_p95_latency(requests: List[ToyServingRequest]) -> Optional[float]:
    rows = rows_to_df(summarize_requests(requests))
    admitted = rows[(rows["rejected"] == False) & (rows["end_to_end_steps"].notna())]
    values = [float(x) for x in admitted["end_to_end_steps"].tolist()]
    return nearest_rank(values, 0.95)

comparison_df = pd.DataFrame([
    {
        "policy": "blind_queueing",
        "admitted_requests": sum(1 for req in blind_results if not req.rejected and req.finish_step is not None),
        "rejected_requests": sum(1 for req in blind_results if req.rejected),
        "admitted_p95_end_to_end": admitted_p95_latency(blind_results),
    },
    {
        "policy": "protective_admission",
        "admitted_requests": sum(1 for req in protected_results if not req.rejected and req.finish_step is not None),
        "rejected_requests": sum(1 for req in protected_results if req.rejected),
        "admitted_p95_end_to_end": admitted_p95_latency(protected_results),
    },
])

print("\nOverload policy comparison:")
print(comparison_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comparison_df["policy"], comparison_df["admitted_p95_end_to_end"])
ax.set_ylabel("admitted p95 end-to-end steps")
ax.set_title("Overload policy comparison")
plt.tight_layout()
plt.show()
plt.close(fig)



## 38. Real advanced interview questions with weak/strong answers

These are the kinds of questions that reveal whether someone understands **serving systems** or only knows isolated kernels.

### Important note on the arithmetic below

Where I use numbers, they are **back-of-the-envelope reasoning** to structure the answer.
They are **not benchmark claims**.



### Question 1: "Your TTFT doubled after launching a heavier RAG workload. Output tokens/sec stayed roughly flat. What do you suspect first?"

**Weak answer:**
> "Attention got slower."

**Problem:** It jumps to a kernel conclusion without checking whether the workload changed on the **prompt side**.

**Strong answer:**
> "I would first suspect prompt-side pressure rather than decode-side pressure.
>
> RAG often increases prompt length and prompt variance, so TTFT can rise even if steady-state decode throughput stays roughly flat.
>
> I would check prompt length distribution, queue time, prefill share of GPU time, whether long prefills are crowding out short interactive requests, and whether prefix reuse opportunities changed.
>
> Then I would test chunked prefill, admission rules for oversized prompts, and whether different traffic classes should share the same serving pool."

**Why this is strong:**  
It separates **TTFT** from aggregate throughput and starts with the most likely workload shift.

---

### Question 2: "We enabled speculative decoding, but p95 ITL improved only a little while GPU usage went up. Why?"

**Weak answer:**
> "The implementation must be buggy."

**Problem:** It blames the implementation before checking the operating regime.

**Strong answer:**
> "Not necessarily. I would first check speculation acceptance rate and the extra draft overhead.
>
> If acceptance is low, we are paying for draft work without skipping enough expensive verifier work.
>
> I would also ask whether the workload is already saturating the target model, because speculation often helps most in the right memory-bound, medium-load regime, not automatically at peak saturation.
>
> Then I would compare different draft lengths, traffic classes, and disable speculation where it is net-negative."

**Why this is strong:**  
It explains a realistic failure mode instead of assuming the feature is broken.

---

### Question 3: "We set `max_num_batched_tokens` very high and throughput improved, but user experience got worse. How can both be true?"

**Weak answer:**
> "That should not happen if throughput improved."

**Problem:** It treats throughput as the only metric that matters.

**Strong answer:**
> "Higher token budgets can increase packing efficiency, but they can also let long prefills occupy more of the step budget and interfere with decode smoothness or short-request TTFT.
>
> So the system can look better in aggregate tokens/sec while p95 TTFT or p95 ITL gets worse for interactive traffic.
>
> I would break the workload down by request class and compare p50/p95 TTFT, ITL, queue time, and admitted-user latency before calling the change a win."

**Why this is strong:**  
It shows that the candidate understands **throughput vs latency-distribution tradeoffs**.

---

### Question 4: "We see preemption warnings and rising tail latency. What should the team investigate first?"

**Weak answer:**
> "We need to optimize the attention kernel."

**Problem:** It ignores the resource that is actually under pressure.

**Strong answer:**
> "I would treat it as a KV-capacity problem first.
>
> Preemption means active sequences are competing for limited live KV space, so the system may be pausing requests and later recomputing their context.
>
> I would inspect context length growth, concurrency, cache hit rate, output length policy, and whether we need more KV-friendly settings such as tighter admission caps, different parallelism, reuse, offload, or shorter outputs.
>
> Only after that would I decide whether the real bottleneck is compute, memory, or capacity planning."

**Why this is strong:**  
It identifies the **constrained resource** before prescribing a fix.

---

### Question 5: "How would you explain the difference between router, launcher, and model server to a junior engineer?"

**Weak answer:**
> "They all help serve the model."

**Problem:** It is technically true but operationally useless.

**Strong answer:**
> "I would separate the steady-state request path from the orchestration path.
>
> The router is in the request hot path: it receives requests, queues them, batches them, and forwards work to model servers.
>
> The model server owns the model execution and the hardware work.
>
> The launcher is mostly about starting and wiring those processes together; it is not the main request hot path once the system is up.
>
> That distinction matters because the bottlenecks and operational failures are different in each layer."

**Why this is strong:**  
It shows **systems decomposition**, not just vocabulary recall.

---

### Question 6: "A model fits after INT4 quantization, but latency got worse. How is that possible?"

**Weak answer:**
> "Quantization should always be faster."

**Problem:** It confuses a memory win with an end-to-end latency win.

**Strong answer:**
> "Smaller weights do not guarantee lower end-to-end latency.
>
> The quantized kernel path may be worse on that hardware, dequant overhead may be meaningful, communication may still dominate, or the workload may be more constrained by KV/cache or scheduler effects than by raw weight bandwidth.
>
> I would separate the memory footprint improvement from the actual latency measurement, verify the kernel path in use, and compare p50/p95 response metrics before declaring success."

**Why this is strong:**  
It treats quantization as a **system tradeoff**, not a slogan.

---

### Question 7: "When would disaggregated prefill/decode help, and when would it hurt?"

**Weak answer:**
> "It always helps because prefill and decode are different."

**Problem:** It ignores transfer cost and operational complexity.

**Strong answer:**
> "It helps when TTFT and ITL want very different resource tuning and the workload is rich enough to justify splitting those phases.
>
> It can let me tune prefill-heavy capacity separately from decode-heavy capacity, which is attractive for long prompts plus interactive streaming.
>
> But it also adds KV transfer cost, more moving parts, and more failure modes.
>
> I would test whether the latency gain survives those costs instead of assuming the separation is automatically worth it."

**Why this is strong:**  
It shows **conditional engineering judgment**.

---

### Question 8: "Prefix reuse hit rate is unexpectedly low even though everyone shares the same system prompt. Why might that happen?"

**Weak answer:**
> "The cache is probably broken."

**Problem:** It ignores how brittle exact-prefix identity really is.

**Strong answer:**
> "I would first verify token-level identity.
>
> Small differences in whitespace, timestamps, tracing fields, retrieved context ordering, or tokenizer versioning can turn visually similar prompts into different token prefixes.
>
> I would inspect the exact tokenized prefix, normalize prompt construction, and verify the reuse key before blaming the runtime layer."

**Why this is strong:**  
It shows you understand that reuse is determined by **exact token identity**, not visual similarity.

---

### Question 9: "Under overload, should we just accept everything and let requests queue?"

**Weak answer:**
> "Yes, rejecting is always worse for users."

**Problem:** It ignores the quality of service for admitted traffic.

**Strong answer:**
> "Not blindly.
>
> If queue growth destroys admitted p95 latency, then we may violate the product SLO for everyone.
>
> I would usually prefer explicit overload behavior: queue caps, traffic shaping, output clamping, long-job isolation, or early rejection with retry guidance.
>
> The goal is to protect the latency of admitted interactive traffic instead of silently degrading the whole system."

**Why this is strong:**  
It shows mature thinking about **SLO protection and graceful degradation**.

---

### Question 10: "Router CPU is pegged, GPU utilization is low, and TTFT is bad. What does that tell you?"

**Weak answer:**
> "The GPUs are too slow."

**Problem:** It assumes the accelerator is the bottleneck without evidence.

**Strong answer:**
> "It suggests the GPUs are underfed.
>
> I would inspect tokenization, request validation, prompt construction, tool / JSON shaping, batching logic, and any front-end queueing before touching CUDA kernels.
>
> If the front-end is the bottleneck, GPU optimization will not fix TTFT.
>
> This is a classic example of end-to-end latency being dominated by non-GPU stages."

**Why this is strong:**  
It recognizes an important infra reality: the accelerator is not always the main bottleneck.



## 39. Final interview takeaways

If you can explain the following clearly, you are in strong shape for LLM inference / serving interviews:

1. **model-side foundations**
   - decoder-only causal masking
   - prefill vs decode
   - what the KV cache stores
   - why cached and uncached decoding should agree semantically
   - why `num_key_value_heads` matters for GQA / MQA cache size

2. **paged KV mental model**
   - dense vs paged cache
   - logical blocks vs physical blocks
   - block tables
   - prefix sharing, refcounts, and copy-on-write

3. **serving control-plane topics**
   - scheduler token budgets
   - admission control
   - queue time, TTFT, ITL, end-to-end latency
   - chunked prefill
   - preemption and recompute under KV pressure
   - host offload / cache reuse
   - speculative decoding
   - quantization tradeoffs
   - disaggregated prefill / decode
   - overload behavior and SLO protection

4. **architecture fluency**
   - router / launcher / model-server mental model
   - API server / engine core / GPU worker mental model
   - where scheduling intelligence lives
   - why the bottleneck is often the system, not just the kernel

That combination is exactly what makes a candidate sound like an **infra engineer** instead of someone who only memorized model terminology.



## 40. Suggested next steps

After this notebook, a very good progression is:

1. re-read the official vLLM scheduler, metrics, and optimization docs with this toy mental model in mind  
2. compare vLLM and one other serving stack at the architecture level  
3. practice explaining TTFT vs ITL vs throughput using small queueing examples  
4. practice answering overload / SLO questions out loud  
5. study one real serving incident pattern:
   - prefix hit rate collapse
   - KV-capacity preemption
   - router CPU bottleneck
   - quantization regression
   - speculation with low acceptance

The most valuable outcome is not memorizing buzzwords.

The valuable outcome is being able to say:

> "Here is the likely constrained resource.  
> Here is how I would measure it.  
> Here is the safest first mitigation.  
> Here is how I would verify whether it actually helped."



## References for further reading

These are the official docs that map most directly to the topics in this notebook:

- vLLM PagedAttention design note
- vLLM Architecture Overview
- vLLM scheduler configuration docs
- vLLM optimization / tuning docs
- vLLM production metrics docs
- vLLM speculative decoding docs
- vLLM disaggregated prefilling docs
- vLLM engine arguments docs for KV offloading

- Hugging Face TGI architecture docs
- Hugging Face TGI metrics docs
- Hugging Face TGI quantization docs

- TensorRT-LLM KV cache system docs
- TensorRT-LLM KV cache reuse docs

This notebook is intentionally educational and toy-sized, but the mental models line up with real serving systems.
